# MLP + numerical-feature Transformer NetFlow domain unlearning

This is a complete, self-contained notebook for leave-one-domain-out machine unlearning. It compares an original model trained on all four NetFlow domains, an approximately unlearned model produced by logged-update rollback plus a short retained-data repair, and a counterfactual reference retrained from the exact same initialization with the selected domain absent. Both requested architectures are included: an MLP and an FT-style numerical feature Transformer, retained under the project-facing name `tabtransformer`.Computation begins in Section 12.

The publication protocol deliberately fixes every choice that could otherwise retain forgotten-domain information outside the model weights. It uses a feature list declared before data loading, a transform with no learned statistics, per-domain sampling seeds, grouped splits, fixed loss weights, a fixed epoch count, and plain SGD without momentum or weight decay. This makes the scratch reference a meaningful “the domain did not participate in the configured learning pipeline” counterfactual. The rollback method is still approximate, and membership attacks are empirical privacy audits rather than proofs of deletion.

## 1. Install dependencies

**What the following block does:** This cell installs the libraries used by every later cell into the active Jupyter kernel. NumPy and pandas handle arrays and tables; SciPy and scikit-learn provide metrics, confidence intervals, and the learned privacy attack; PyTorch defines and trains both neural networks; PyArrow reads Parquet data by row batch; KaggleHub is an optional fallback when the configured files are not local; psutil samples process memory; and joblib stores preprocessing metadata. It does not load data or train anything. Run it once in a fresh environment, and restart the kernel only if Jupyter explicitly asks you to do so.

In [6]:
# %pip install -q "numpy>=1.26,<2" "pandas>=2,<3" "scipy>=1.11,<1.15"     "scikit-learn>=1.4,<1.7" "torch>=2.2,<3" "pyarrow>=15,<24"     "kagglehub>=0.3,<1" "psutil>=5.9,<8" "joblib>=1.3,<2"

In [7]:
!pip install -q polars kagglehub

## 2. Experiment configuration types

**What the following block does:** This cell defines validated dataclasses for datasets, preprocessing, models, training, unlearning, privacy attacks, and runtime behavior. It also declares the 48 behavioural inputs used for the standardized NF-v3 corpora before any file is inspected; addresses, ports, and absolute timestamps are intentionally absent, while `FLOW_DURATION` and `BYTES_PER_PKT` are deterministic derived features. Strict mode rejects choices that would weaken the counterfactual comparison, including a data-derived schema, a learned scaler, row-wise splitting, AdamW, SGD momentum, weight decay, balanced domain resampling, early checkpoint selection, or fewer than three publication seeds. These are definitions only. Students normally edit the control panel in Section 11 rather than this cell.

In [8]:
"""Typed JSON configuration for the NetFlow unlearning experiment."""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

# Publicly specified NF-v3 behavioural fields.  The list is fixed before any
# experiment is loaded, so a forgotten domain cannot influence feature
# selection.  Direct identifiers, ports, and absolute timestamps are excluded;
# FLOW_DURATION and BYTES_PER_PKT are deterministic derived fields.
PUBLIC_V3_MODEL_FEATURES = [
    "PROTOCOL",
    "L7_PROTO",
    "IN_BYTES",
    "OUT_BYTES",
    "IN_PKTS",
    "OUT_PKTS",
    "FLOW_DURATION",
    "TCP_FLAGS",
    "CLIENT_TCP_FLAGS",
    "SERVER_TCP_FLAGS",
    "DURATION_IN",
    "DURATION_OUT",
    "MIN_TTL",
    "MAX_TTL",
    "LONGEST_FLOW_PKT",
    "SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN",
    "MAX_IP_PKT_LEN",
    "SRC_TO_DST_SECOND_BYTES",
    "DST_TO_SRC_SECOND_BYTES",
    "RETRANSMITTED_IN_BYTES",
    "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES",
    "RETRANSMITTED_OUT_PKTS",
    "SRC_TO_DST_AVG_THROUGHPUT",
    "DST_TO_SRC_AVG_THROUGHPUT",
    "NUM_PKTS_UP_TO_128_BYTES",
    "NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES",
    "NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES",
    "TCP_WIN_MAX_IN",
    "TCP_WIN_MAX_OUT",
    "ICMP_TYPE",
    "ICMP_IPV4_TYPE",
    "DNS_QUERY_ID",
    "DNS_QUERY_TYPE",
    "DNS_TTL_ANSWER",
    "FTP_COMMAND_RET_CODE",
    "SRC_TO_DST_IAT_MIN",
    "SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_AVG",
    "SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN",
    "DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_AVG",
    "DST_TO_SRC_IAT_STDDEV",
    "BYTES_PER_PKT",
]


@dataclass
class DatasetConfig:
    """One standardized NetFlow domain.

    ``path`` may name a CSV/Parquet file, a directory, or a glob.  When it does
    not resolve locally, ``kaggle_slug`` is used (if downloads are enabled).
    ``file_pattern`` is applied inside a directory or downloaded Kaggle folder;
    it is especially useful when one download contains multiple datasets.
    """

    name: str
    path: str | None = None
    kaggle_slug: str | None = None
    file_pattern: str = "**/*"
    label_column: str | None = None
    sample_rows: int | None = None
    encoding: str = "utf-8"
    allow_multiple_files: bool = False

    def validate(self) -> None:
        if not self.name.strip():
            raise ValueError("Every dataset needs a non-empty name")
        if not self.path and not self.kaggle_slug:
            raise ValueError(
                f"Dataset {self.name!r} needs either 'path' or 'kaggle_slug'"
            )
        if self.sample_rows is not None and self.sample_rows < 10:
            raise ValueError(f"{self.name}: sample_rows must be >= 10 or null")
        if not self.encoding.strip():
            raise ValueError(f"{self.name}: encoding must be non-empty")


@dataclass
class DataConfig:
    datasets: list[DatasetConfig]
    common_features: list[str] | None = None
    train_fraction: float = 0.70
    validation_fraction: float = 0.15
    test_fraction: float = 0.15
    seed: int = 42
    sample_rows_per_dataset: int | None = 250_000
    csv_chunk_rows: int = 100_000
    scaler: str = "fixed_log"
    scaler_fit_rows: int | None = 200_000
    allow_kaggle_download: bool = True
    split_strategy: str = "group_stratified"
    group_columns: list[str] = field(
        default_factory=lambda: ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "PROTOCOL"]
    )
    drop_exact_duplicates: bool = True
    hash_source_files: bool = True
    strict_protocol: bool = True

    def validate(self) -> None:
        if not 4 <= len(self.datasets) <= 5:
            raise ValueError(
                "The requested protocol needs 4 or 5 datasets; "
                f"configuration contains {len(self.datasets)}"
            )
        names = [d.name for d in self.datasets]
        if len(set(names)) != len(names):
            raise ValueError(f"Dataset names must be unique: {names}")
        for dataset in self.datasets:
            dataset.validate()
        total = self.train_fraction + self.validation_fraction + self.test_fraction
        if abs(total - 1.0) > 1e-9:
            raise ValueError("train/validation/test fractions must add to 1")
        if min(self.train_fraction, self.validation_fraction, self.test_fraction) <= 0:
            raise ValueError("All split fractions must be positive")
        if self.csv_chunk_rows < 1_000:
            raise ValueError("csv_chunk_rows must be at least 1000")
        if (
            self.sample_rows_per_dataset is not None
            and self.sample_rows_per_dataset < 10
        ):
            raise ValueError("sample_rows_per_dataset must be >= 10 or null")
        if self.common_features is not None:
            normalized = [feature.strip().upper() for feature in self.common_features]
            if not normalized or any(not feature for feature in normalized):
                raise ValueError("common_features must contain non-empty names")
            if len(set(normalized)) != len(normalized):
                raise ValueError("common_features must not contain duplicates")
        if self.scaler not in {
            "fixed_log",
            "quantile",
            "robust",
            "standard",
            "minmax",
            "none",
        }:
            raise ValueError(
                "scaler must be one of: fixed_log, quantile, robust, standard, "
                "minmax, none"
            )
        if self.split_strategy not in {"group_stratified", "row_stratified"}:
            raise ValueError(
                "split_strategy must be 'group_stratified' or 'row_stratified'"
            )
        normalized_groups = [column.strip().upper() for column in self.group_columns]
        if self.split_strategy == "group_stratified" and not normalized_groups:
            raise ValueError("group_stratified splitting needs group_columns")
        if self.strict_protocol:
            if self.common_features is None:
                raise ValueError(
                    "Strict protocol requires a predeclared common_features list"
                )
            if self.scaler != "fixed_log":
                raise ValueError(
                    "Strict protocol requires the stateless fixed_log transform"
                )
            if self.split_strategy != "group_stratified":
                raise ValueError("Strict protocol requires group_stratified splitting")


@dataclass
class ModelConfig:
    architectures: list[str] = field(default_factory=lambda: ["mlp", "tabtransformer"])
    latent_dim: int = 32
    mlp_hidden_dims: list[int] = field(default_factory=lambda: [128, 64])
    dropout: float = 0.10
    tab_d_token: int = 16
    tab_heads: int = 4
    tab_layers: int = 2
    tab_ffn_factor: int = 4

    def validate(self) -> None:
        allowed = {"mlp", "tabtransformer"}
        if not self.architectures or set(self.architectures) - allowed:
            raise ValueError(f"architectures must be a non-empty subset of {allowed}")
        if self.latent_dim < 2 or any(width < 2 for width in self.mlp_hidden_dims):
            raise ValueError("latent and hidden dimensions must be >= 2")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1)")
        if self.tab_d_token < 1 or self.tab_heads < 1 or self.tab_ffn_factor < 1:
            raise ValueError(
                "TabTransformer token/head/FFN dimensions must be positive"
            )
        if self.tab_d_token % self.tab_heads:
            raise ValueError("tab_d_token must be divisible by tab_heads")
        if self.tab_layers < 1:
            raise ValueError("tab_layers must be >= 1")


@dataclass
class TrainingConfig:
    epochs: int = 20
    batch_size: int = 256
    learning_rate: float = 1e-3
    optimizer: str = "sgd"
    sgd_momentum: float = 0.0
    weight_decay: float = 0.0
    patience: int = 5
    min_delta: float = 1e-4
    fixed_class_weights: list[float] = field(default_factory=lambda: [1.0, 1.0])
    gradient_clip_norm: float = 5.0
    domain_sampling: str = "proportional"
    checkpoint_selection: str = "final"
    seeds: list[int] = field(default_factory=lambda: [42])
    strict_unlearning_protocol: bool = True

    def validate(self) -> None:
        if min(self.epochs, self.batch_size, self.patience) < 1:
            raise ValueError("epochs, batch_size, and patience must be positive")
        if self.learning_rate <= 0 or self.weight_decay < 0:
            raise ValueError("Invalid optimizer hyperparameters")
        if self.optimizer not in {"sgd", "adamw"}:
            raise ValueError("optimizer must be 'sgd' or 'adamw'")
        if not 0 <= self.sgd_momentum < 1:
            raise ValueError("sgd_momentum must be in [0, 1)")
        if self.min_delta < 0:
            raise ValueError("min_delta must be non-negative")
        if len(self.fixed_class_weights) != 2 or any(
            value <= 0 for value in self.fixed_class_weights
        ):
            raise ValueError("fixed_class_weights must contain two positive values")
        if self.gradient_clip_norm < 0:
            raise ValueError("gradient_clip_norm must be non-negative")
        if self.domain_sampling not in {"proportional", "balanced"}:
            raise ValueError("domain_sampling must be 'proportional' or 'balanced'")
        if self.checkpoint_selection not in {"final", "early_stopping"}:
            raise ValueError("checkpoint_selection must be 'final' or 'early_stopping'")
        if not self.seeds:
            raise ValueError("At least one seed is required")
        if len(set(self.seeds)) != len(self.seeds):
            raise ValueError("Training seeds must be unique")
        if self.strict_unlearning_protocol:
            if self.optimizer != "sgd":
                raise ValueError("Strict unlearning requires optimizer='sgd'")
            if self.sgd_momentum != 0 or self.weight_decay != 0:
                raise ValueError(
                    "Strict unlearning requires zero momentum and weight decay"
                )
            if self.domain_sampling != "proportional":
                raise ValueError(
                    "Strict unlearning requires proportional domain sampling"
                )
            if self.checkpoint_selection != "final":
                raise ValueError("Strict unlearning requires a fixed final checkpoint")


@dataclass
class UnlearningConfig:
    method: str = "amnesiac_rollback_repair"
    rollback_scale: float = 1.0
    repair_epochs: int = 2
    repair_fraction: float = 0.25
    repair_learning_rate: float = 2e-4

    def validate(self) -> None:
        if self.method != "amnesiac_rollback_repair":
            raise ValueError("Only 'amnesiac_rollback_repair' is implemented")
        if self.rollback_scale < 0:
            raise ValueError("rollback_scale must be non-negative")
        if self.repair_epochs < 0:
            raise ValueError("repair_epochs must be >= 0")
        if not 0 < self.repair_fraction <= 1:
            raise ValueError("repair_fraction must be in (0, 1]")
        if self.repair_learning_rate <= 0:
            raise ValueError("repair_learning_rate must be positive")


@dataclass
class AttackConfig:
    max_samples_per_class: int = 10_000
    fixed_fpr: float = 0.01
    bootstrap_repetitions: int = 1_000
    confidence_level: float = 0.95

    def validate(self) -> None:
        if self.max_samples_per_class < 10:
            raise ValueError("max_samples_per_class must be >= 10")
        if not 0 < self.fixed_fpr < 1:
            raise ValueError("fixed_fpr must be in (0, 1)")
        if self.bootstrap_repetitions < 0:
            raise ValueError("bootstrap_repetitions must be >= 0")
        if not 0 < self.confidence_level < 1:
            raise ValueError("confidence_level must be in (0, 1)")


@dataclass
class RuntimeConfig:
    output_dir: str = "artifacts/netflow_unlearning"
    device: str = "auto"
    num_workers: int = 0
    save_predictions: bool = False
    deterministic: bool = True

    def validate(self) -> None:
        if self.device not in {"auto", "cpu", "cuda", "mps"}:
            raise ValueError("device must be auto, cpu, cuda, or mps")
        if self.num_workers < 0:
            raise ValueError("num_workers must be >= 0")


@dataclass
class ExperimentConfig:
    data: DataConfig
    models: ModelConfig = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    unlearning: UnlearningConfig = field(default_factory=UnlearningConfig)
    attack: AttackConfig = field(default_factory=AttackConfig)
    runtime: RuntimeConfig = field(default_factory=RuntimeConfig)

    def validate(self) -> None:
        self.data.validate()
        self.models.validate()
        self.training.validate()
        self.unlearning.validate()
        self.attack.validate()
        self.runtime.validate()
        if self.data.strict_protocol != self.training.strict_unlearning_protocol:
            raise ValueError(
                "data.strict_protocol and training.strict_unlearning_protocol "
                "must agree"
            )
        if self.data.strict_protocol and len(self.training.seeds) < 3:
            raise ValueError(
                "Publication protocol requires at least three independent seeds"
            )

    @classmethod
    def from_dict(cls, raw: dict[str, Any]) -> ExperimentConfig:
        data_raw = dict(raw["data"])
        data_raw["datasets"] = [DatasetConfig(**d) for d in data_raw["datasets"]]
        config = cls(
            data=DataConfig(**data_raw),
            models=ModelConfig(**raw.get("models", {})),
            training=TrainingConfig(**raw.get("training", {})),
            unlearning=UnlearningConfig(**raw.get("unlearning", {})),
            attack=AttackConfig(**raw.get("attack", {})),
            runtime=RuntimeConfig(**raw.get("runtime", {})),
        )
        config.validate()
        return config

    @classmethod
    def from_json(cls, path: str | Path) -> ExperimentConfig:
        config_path = Path(path).expanduser().resolve()
        with config_path.open("r", encoding="utf-8") as handle:
            raw = json.load(handle)
        config = cls.from_dict(raw)

        # Dataset paths in a checked-in config are relative to that config.
        for dataset in config.data.datasets:
            if dataset.path and not Path(dataset.path).expanduser().is_absolute():
                dataset.path = str((config_path.parent / dataset.path).resolve())
        output = Path(config.runtime.output_dir).expanduser()
        if not output.is_absolute():
            config.runtime.output_dir = str((config_path.parent / output).resolve())
        return config

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    def to_json(self, path: str | Path) -> None:
        destination = Path(path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("w", encoding="utf-8") as handle:
            json.dump(self.to_dict(), handle, indent=2, sort_keys=True)


## 3. Data loading, labels, features, provenance, and splits

**What the following block does:** This cell implements the complete data pipeline. It resolves one explicit CSV or Parquet source per domain, refusing multiple matches unless the user declares that they are intentional shards; reads large files in chunks with an explicit encoding; takes a reproducible uniform row sample; records the source file and original row number; removes exact duplicate records; canonicalizes columns; and converts common label conventions to `0 = benign` and `1 = attack`. It deterministically derives duration and bytes-per-packet, requires every predeclared feature to be present and numeric, and applies the same data-independent formula, `tanh(sign(x) × log1p(abs(x)) / 10)`, to every value. Because this transform is not fitted, a forgotten domain cannot remain in scaler parameters.

The cell then forms each domain's 70/15/15 partitions with a deterministic stratified group assignment based on the bidirectional endpoint pair and protocol. All flows in one group stay in one partition, reducing duplicate-host or repeated-connection leakage that a random row split would permit. Sampling and split seeds are derived from the domain name, so removing another domain does not change retained records. The saved manifest includes file size, modification time, optional SHA-256, duplicate counts, class rates, sampled-record fingerprint, split indices, group IDs, source-record IDs, and labels. Automatic feature intersection and learned scalers remain available only for explicitly non-strict exploratory work.

In [9]:
"""NetFlow loading, common-schema construction, and leakage-safe splitting."""

from __future__ import annotations

import glob
import hashlib
import json
import math
import re
import warnings
from collections.abc import Iterable, Iterator, Sequence
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SUPPORTED_SUFFIXES = {".csv", ".parquet", ".pq"}
LABEL_ALIASES = ("LABEL", "BINARY_LABEL", "TARGET", "CLASS")
BENIGN_TOKENS = {
    "0",
    "benign",
    "normal",
    "normal.",
    "background",
    "legitimate",
    "false",
    "no",
    "nonattack",
    "non_attack",
    "non-attack",
}

# Identifiers make domain recognition easy without describing flow behaviour.
# This is the same exclusion policy used by the existing notebooks, expanded
# with common spelling variants.
EXCLUDED_COLUMNS = {
    "ATTACK",
    "ATTACK_CAT",
    "ATTACK_CATEGORY",
    "DATASET",
    "DATE",
    "TIMESTAMP",
    "FLOW_ID",
    "ROW_ID",
    "ID",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "IPV6_SRC_ADDR",
    "IPV6_DST_ADDR",
    "SRC_IP",
    "DST_IP",
    "SOURCE_IP",
    "DESTINATION_IP",
    "L4_SRC_PORT",
    "L4_DST_PORT",
    "SRC_PORT",
    "DST_PORT",
    "SOURCE_PORT",
    "DESTINATION_PORT",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "FLOW_DURATION_MILLISECONDS",
    "NF_SOURCE_FILE",
    "NF_SOURCE_ROW",
}


def canonical_name(value: object) -> str:
    """Return a stable, case-insensitive feature name."""

    name = re.sub(r"[^A-Z0-9]+", "_", str(value).strip().upper()).strip("_")
    if not name:
        raise ValueError(f"Empty column name after canonicalization: {value!r}")
    return name


def _domain_seed(base_seed: int, domain_name: str, purpose: str) -> int:
    """Derive a seed unaffected by dataset ordering or leave-one-out removal."""

    payload = f"{base_seed}\x1f{domain_name}\x1f{purpose}".encode()
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little")


def _canonicalize_frame(frame: pd.DataFrame) -> pd.DataFrame:
    names = [canonical_name(column) for column in frame.columns]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"Columns collide after canonicalization: {duplicates}")
    result = frame.copy()
    result.columns = names
    return result


def _files_below(root: Path, pattern: str) -> list[Path]:
    return sorted(
        {
            path.resolve()
            for path in root.glob(pattern)
            if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
        }
    )


def resolve_dataset_files(
    spec: DatasetConfig, allow_kaggle_download: bool = True
) -> list[Path]:
    """Resolve local data first and optionally fall back to Kaggle.

    No download is attempted when at least one matching local file exists.
    """

    files: list[Path] = []
    if spec.path:
        expanded = str(Path(spec.path).expanduser())
        if any(character in expanded for character in "*?["):
            files = sorted(
                Path(match).resolve()
                for match in glob.glob(expanded, recursive=True)
                if Path(match).is_file()
                and Path(match).suffix.lower() in SUPPORTED_SUFFIXES
            )
        else:
            candidate = Path(expanded)
            if candidate.is_file() and candidate.suffix.lower() in SUPPORTED_SUFFIXES:
                files = [candidate.resolve()]
            elif candidate.is_dir():
                files = _files_below(candidate, spec.file_pattern)

    if not files and spec.kaggle_slug and allow_kaggle_download:
        try:
            import kagglehub  # type: ignore
        except ImportError as exc:
            raise RuntimeError(
                f"No local files found for {spec.name!r}. Install kagglehub or "
                "put the dataset at the configured local path."
            ) from exc
        downloaded = Path(kagglehub.dataset_download(spec.kaggle_slug))
        files = _files_below(downloaded, spec.file_pattern)

    if not files:
        local_hint = f" at {spec.path!r}" if spec.path else ""
        download_hint = (
            f" (Kaggle fallback {spec.kaggle_slug!r})" if spec.kaggle_slug else ""
        )
        raise FileNotFoundError(
            f"No CSV/Parquet files found for {spec.name!r}{local_hint}{download_hint}."
        )
    if len(files) > 1 and not spec.allow_multiple_files:
        preview = [str(path) for path in files[:8]]
        raise ValueError(
            f"{spec.name!r} matched {len(files)} files, which could silently "
            "concatenate mirrors or duplicate exports. Narrow file_pattern, or "
            "set allow_multiple_files=True only when these are intentional shards. "
            f"Matches: {preview}"
        )
    return files


def _iter_file_chunks(
    path: Path, csv_chunk_rows: int, encoding: str = "utf-8"
) -> Iterator[pd.DataFrame]:
    if path.suffix.lower() == ".csv":
        # Encoding is explicit.  Retrying a partially-consumed generator with a
        # second encoding can duplicate every chunk yielded before a late decode
        # error, so a silent fallback is deliberately forbidden.
        yield from pd.read_csv(
            path,
            chunksize=csv_chunk_rows,
            low_memory=False,
            on_bad_lines="error",
            encoding=encoding,
        )
        return

    # Prefer row-group iteration so a large Parquet file is not materialized at
    # once.  pandas remains a fallback for environments without pyarrow.
    try:
        import pyarrow.parquet as pq  # type: ignore

        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(batch_size=csv_chunk_rows):
            yield batch.to_pandas()
    except ImportError:
        yield pd.read_parquet(path)


def _uniform_sample_files(
    files: Sequence[Path],
    limit: int | None,
    csv_chunk_rows: int,
    seed: int,
    encoding: str = "utf-8",
) -> pd.DataFrame:
    """Read files with an exact bounded uniform sample.

    Every row receives an i.i.d. random priority and the smallest ``limit``
    priorities are retained.  Unlike taking the first N rows, this does not
    bias a time-ordered NetFlow file, and memory is O(limit + chunk size).
    """

    rng = np.random.default_rng(seed)
    priority_column = "__NF_UNLEARNING_SAMPLE_PRIORITY__"
    reservoir: pd.DataFrame | None = None
    observed = 0

    if limit is None:
        warnings.warn(
            "sample_rows=None materializes the complete corpus in memory; use a "
            "bounded, predeclared sample for ordinary experiments.",
            ResourceWarning,
            stacklevel=2,
        )

    for path in files:
        file_row_offset = 0
        for chunk in _iter_file_chunks(path, csv_chunk_rows, encoding=encoding):
            if chunk.empty:
                continue
            if priority_column in chunk.columns:
                raise ValueError(f"Reserved column exists in {path}: {priority_column}")
            reserved = {"NF_SOURCE_FILE", "NF_SOURCE_ROW"} & {
                canonical_name(column) for column in chunk.columns
            }
            if reserved:
                raise ValueError(
                    f"Reserved provenance columns exist in {path}: {reserved}"
                )
            chunk = chunk.copy()
            chunk["NF_SOURCE_FILE"] = str(path)
            chunk["NF_SOURCE_ROW"] = np.arange(
                file_row_offset, file_row_offset + len(chunk), dtype=np.int64
            )
            file_row_offset += len(chunk)
            observed += len(chunk)
            if limit is None:
                reservoir = (
                    chunk
                    if reservoir is None
                    else pd.concat([reservoir, chunk], ignore_index=True, sort=False)
                )
                continue

            priorities = rng.random(len(chunk))
            if len(chunk) > limit:
                keep = np.argpartition(priorities, limit - 1)[:limit]
                chunk = chunk.iloc[keep].copy()
                priorities = priorities[keep]
            chunk[priority_column] = priorities
            reservoir = (
                chunk
                if reservoir is None
                else pd.concat([reservoir, chunk], ignore_index=True, sort=False)
            )
            if len(reservoir) > 2 * limit:
                reservoir = reservoir.nsmallest(limit, priority_column).copy()

    if reservoir is None or reservoir.empty:
        raise ValueError(
            f"Dataset files contain no rows: {[str(path) for path in files]}"
        )
    if limit is not None:
        reservoir = reservoir.nsmallest(min(limit, len(reservoir)), priority_column)
        reservoir = reservoir.drop(columns=[priority_column])
    reservoir = reservoir.reset_index(drop=True)
    reservoir.attrs["rows_observed"] = observed
    return reservoir


def _find_label_column(frame: pd.DataFrame, requested: str | None) -> str:
    if requested:
        candidate = canonical_name(requested)
        if candidate not in frame.columns:
            raise ValueError(
                f"Configured label column {requested!r} is absent. "
                f"Columns include: {list(frame.columns)[:20]}"
            )
        return candidate
    for candidate in LABEL_ALIASES:
        if candidate in frame.columns:
            return candidate
    if "ATTACK" in frame.columns:
        return "ATTACK"
    raise ValueError(
        "No binary label found. Set label_column in the dataset configuration."
    )


def _binary_labels(
    series: pd.Series,
    dataset_name: str,
    require_zero_one_numeric: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    """Normalize common binary/named labels and return labels plus valid mask."""

    valid = ~series.isna()
    if not valid.any():
        raise ValueError(f"{dataset_name}: every label is missing")
    clean = series[valid]

    numeric = pd.to_numeric(clean, errors="coerce")
    if numeric.notna().all():
        values = np.sort(numeric.unique())
        if set(values.tolist()).issubset({0, 1}):
            encoded = numeric.astype(np.int64).to_numpy()
        elif require_zero_one_numeric:
            raise ValueError(
                f"{dataset_name}: strict protocol requires numeric labels encoded "
                f"as 0=benign and 1=attack; found {values[:10].tolist()}"
            )
        elif len(values) == 2:
            encoded = (numeric.to_numpy() == values[-1]).astype(np.int64)
        elif 0 in values:
            # Some corpora encode attack families as positive integers.
            encoded = (numeric.to_numpy() != 0).astype(np.int64)
        else:
            raise ValueError(
                f"{dataset_name}: numeric label has no benign zero and is not "
                f"binary (values start {values[:10].tolist()})"
            )
    else:
        normalized = clean.astype(str).str.strip().str.lower()
        benign_mask = normalized.isin(BENIGN_TOKENS)
        if not benign_mask.any():
            examples = sorted(normalized.unique().tolist())[:10]
            raise ValueError(
                f"{dataset_name}: cannot identify a benign label among {examples}"
            )
        encoded = (~benign_mask).astype(np.int64).to_numpy()

    labels = np.full(len(series), -1, dtype=np.int64)
    labels[np.flatnonzero(valid.to_numpy())] = encoded
    valid_mask = labels >= 0
    classes, counts = np.unique(labels[valid_mask], return_counts=True)
    if classes.tolist() != [0, 1]:
        raise ValueError(
            f"{dataset_name}: both benign (0) and attack (1) are required; "
            f"found {dict(zip(classes.tolist(), counts.tolist(), strict=True))}"
        )
    if counts.min() < 3:
        raise ValueError(
            f"{dataset_name}: each class needs at least 3 rows for train/val/test"
        )
    return labels[valid_mask], valid_mask


def _numeric_features(
    frame: pd.DataFrame,
    label_column: str,
    dataset_name: str,
    requested_features: Sequence[str] | None = None,
) -> pd.DataFrame:
    frame = frame.copy()

    # Normalize the duration name across NetFlow releases.  Prefer the direct
    # duration feature; otherwise derive it before timestamps are removed.
    if "FLOW_DURATION" not in frame.columns:
        if "FLOW_DURATION_MILLISECONDS" in frame.columns:
            frame["FLOW_DURATION"] = frame["FLOW_DURATION_MILLISECONDS"]
        elif {
            "FLOW_START_MILLISECONDS",
            "FLOW_END_MILLISECONDS",
        }.issubset(frame.columns):
            start = pd.to_numeric(frame["FLOW_START_MILLISECONDS"], errors="coerce")
            end = pd.to_numeric(frame["FLOW_END_MILLISECONDS"], errors="coerce")
            frame["FLOW_DURATION"] = (end - start).clip(lower=0)

    if {"IN_BYTES", "IN_PKTS"}.issubset(frame.columns):
        in_bytes = pd.to_numeric(frame["IN_BYTES"], errors="coerce")
        in_packets = pd.to_numeric(frame["IN_PKTS"], errors="coerce")
        frame["BYTES_PER_PKT"] = in_bytes / (in_packets + 1e-5)

    excluded = set(EXCLUDED_COLUMNS) | set(LABEL_ALIASES) | {label_column}
    if requested_features is not None:
        candidates = [canonical_name(column) for column in requested_features]
        missing = sorted(set(candidates) - set(frame.columns))
        if missing:
            raise ValueError(
                f"{dataset_name}: predeclared model features are missing: {missing}"
            )
        forbidden = sorted(set(candidates) & excluded)
        if forbidden:
            raise ValueError(
                f"{dataset_name}: requested features contain identifiers or labels: "
                f"{forbidden}"
            )
    else:
        candidates = [column for column in frame.columns if column not in excluded]
    numeric: dict[str, pd.Series] = {}
    dropped: list[str] = []
    for column in candidates:
        converted = pd.to_numeric(frame[column], errors="coerce")
        original_nonmissing = int(frame[column].notna().sum())
        convertible = int(converted.notna().sum())
        ratio = convertible / max(original_nonmissing, 1)
        if ratio >= 0.99:
            numeric[column] = converted
        elif requested_features is not None:
            raise ValueError(
                f"{dataset_name}: predeclared feature {column!r} is only "
                f"{ratio:.1%} numeric; fix the source data instead of selecting "
                "features from observed values"
            )
        else:
            dropped.append(column)

    if dropped:
        warnings.warn(
            f"{dataset_name}: dropping nonnumeric columns {dropped}",
            RuntimeWarning,
            stacklevel=2,
        )
    if not numeric:
        raise ValueError(f"{dataset_name}: no numeric NetFlow features remain")

    result = pd.DataFrame(numeric, index=frame.index)
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return result


@dataclass
class DomainSplit:
    name: str
    x_train: np.ndarray
    y_train: np.ndarray
    x_val: np.ndarray
    y_val: np.ndarray
    x_test: np.ndarray
    y_test: np.ndarray
    train_indices: np.ndarray
    val_indices: np.ndarray
    test_indices: np.ndarray
    train_record_ids: np.ndarray | None = None
    val_record_ids: np.ndarray | None = None
    test_record_ids: np.ndarray | None = None
    train_group_ids: np.ndarray | None = None
    val_group_ids: np.ndarray | None = None
    test_group_ids: np.ndarray | None = None

    @property
    def X_train(self) -> np.ndarray:  # compatibility with notebook notation
        return self.x_train

    @property
    def X_val(self) -> np.ndarray:
        return self.x_val

    @property
    def X_test(self) -> np.ndarray:
        return self.x_test


@dataclass
class PreparedData:
    domains: dict[str, DomainSplit]
    common_features: list[str]
    scaler: FeatureScaler
    source_files: dict[str, list[str]]
    rows_observed: dict[str, int]
    source_metadata: dict[str, list[dict[str, object]]]
    sample_fingerprints: dict[str, str]
    duplicates_removed: dict[str, int]

    @property
    def input_dim(self) -> int:
        return len(self.common_features)


def _stratified_indices(
    labels: np.ndarray,
    train_fraction: float,
    validation_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    partitions: dict[str, list[np.ndarray]] = {"train": [], "val": [], "test": []}
    for label in (0, 1):
        indices = np.flatnonzero(labels == label)
        rng.shuffle(indices)
        count = len(indices)
        train_count = max(1, math.floor(count * train_fraction))
        val_count = max(1, math.floor(count * validation_fraction))
        if train_count + val_count >= count:
            # The earlier class-count validation guarantees at least three.
            train_count, val_count = count - 2, 1
        partitions["train"].append(indices[:train_count])
        partitions["val"].append(indices[train_count : train_count + val_count])
        partitions["test"].append(indices[train_count + val_count :])

    outputs: list[np.ndarray] = []
    for key in ("train", "val", "test"):
        joined = np.concatenate(partitions[key]).astype(np.int64, copy=False)
        rng.shuffle(joined)
        outputs.append(joined)
    return outputs[0], outputs[1], outputs[2]


def _make_group_ids(frame: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
    """Create stable record groups used to keep related flows in one split."""

    normalized = [canonical_name(column) for column in columns]
    missing = sorted(set(normalized) - set(frame.columns))
    if missing:
        raise ValueError(f"Configured group columns are missing: {missing}")
    pieces = [frame[column].fillna("<NA>").astype(str) for column in normalized]
    if normalized[:2] == ["IPV4_SRC_ADDR", "IPV4_DST_ADDR"]:
        # Treat the two directions of the same endpoint pair as one conversation
        # group; otherwise A->B could be in training while B->A is in test.
        source = pieces[0].to_numpy(dtype=str)
        destination = pieces[1].to_numpy(dtype=str)
        first = np.where(source <= destination, source, destination)
        second = np.where(source <= destination, destination, source)
        pieces[:2] = [
            pd.Series(first, index=frame.index),
            pd.Series(second, index=frame.index),
        ]
    group = pieces[0]
    for piece in pieces[1:]:
        group = group.str.cat(piece, sep="|")
    return group.to_numpy(dtype=str)


def _stratified_group_indices(
    labels: np.ndarray,
    groups: np.ndarray,
    train_fraction: float,
    validation_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Split whole groups while approximately preserving size and class rate."""

    labels = np.asarray(labels, dtype=np.int64)
    groups = np.asarray(groups)
    if len(labels) != len(groups):
        raise ValueError("labels and groups have different lengths")
    unique_groups, group_number = np.unique(groups, return_inverse=True)
    if len(unique_groups) < 3:
        raise ValueError("At least three distinct groups are required")

    fractions = np.array(
        [
            train_fraction,
            validation_fraction,
            1.0 - train_fraction - validation_fraction,
        ],
        dtype=np.float64,
    )
    group_class_counts = np.zeros((len(unique_groups), 2), dtype=np.int64)
    np.add.at(group_class_counts, (group_number, labels), 1)
    group_sizes = group_class_counts.sum(axis=1)
    class_totals = group_class_counts.sum(axis=0)
    target_class_counts = fractions[:, None] * class_totals[None, :]
    target_row_counts = fractions * len(labels)
    target_rate = float(labels.mean())
    best: tuple[float, tuple[np.ndarray, np.ndarray, np.ndarray]] | None = None

    # A greedy group assignment is linear in the number of records and avoids
    # the very high cost of repeatedly constructing many StratifiedGroupKFold
    # folds on million-row corpora. Random restarts vary equal-sized group order;
    # the best legal assignment is retained.
    rng = np.random.default_rng(seed)
    # One greedy pass is sufficient for large corpora (and keeps runtime
    # linear); a few random tie-order restarts improve small teaching datasets.
    restart_count = 8 if len(unique_groups) <= 1_000 else 1
    for _ in range(restart_count):
        jitter = rng.random(len(unique_groups))
        order = np.lexsort((jitter, -group_sizes))
        assigned_class_counts = np.zeros((3, 2), dtype=np.float64)
        assigned_row_counts = np.zeros(3, dtype=np.float64)
        group_assignment = np.full(len(unique_groups), -1, dtype=np.int8)
        for current_group in order:
            costs = []
            for split_number in range(3):
                proposed_classes = assigned_class_counts.copy()
                proposed_rows = assigned_row_counts.copy()
                proposed_classes[split_number] += group_class_counts[current_group]
                proposed_rows[split_number] += group_sizes[current_group]
                class_error = np.square(
                    (proposed_classes - target_class_counts)
                    / np.maximum(target_class_counts, 1.0)
                ).sum()
                row_error = np.square(
                    (proposed_rows - target_row_counts)
                    / np.maximum(target_row_counts, 1.0)
                ).sum()
                costs.append(float(class_error + 0.25 * row_error))
            minimum = min(costs)
            choices = np.flatnonzero(np.isclose(costs, minimum))
            selected_split = int(rng.choice(choices))
            group_assignment[current_group] = selected_split
            assigned_class_counts[selected_split] += group_class_counts[current_group]
            assigned_row_counts[selected_split] += group_sizes[current_group]

        candidate = tuple(
            np.flatnonzero(group_assignment[group_number] == split_number).astype(
                np.int64, copy=False
            )
            for split_number in range(3)
        )
        if any(
            len(part) == 0
            or set(np.unique(labels[part]).tolist()) != {0, 1}
            or np.bincount(labels[part], minlength=2).min() < 2
            for part in candidate
        ):
            continue
        sizes = np.array([len(part) / len(labels) for part in candidate])
        rates = np.array([labels[part].mean() for part in candidate])
        score = float(np.abs(sizes - fractions).sum())
        score += float(0.25 * np.abs(rates - target_rate).sum())
        if best is None or score < best[0]:
            best = (score, candidate)

    if best is None:
        raise ValueError(
            "Could not form grouped train/validation/test partitions containing "
            "at least two rows from each class. Choose stronger group columns or "
            "increase the sample size."
        )
    rng = np.random.default_rng(seed)
    outputs = []
    for part in best[1]:
        result = part.copy()
        rng.shuffle(result)
        outputs.append(result)
    return outputs[0], outputs[1], outputs[2]


class FeatureScaler:
    """Feature transform with a publication-safe stateless option.

    ``fixed_log`` applies ``tanh(sign(x) * log1p(abs(x)) / 10)`` using no fitted
    data.  Therefore the preprocessing state is identical whether or not any
    domain exists.  Learned scalers remain available only for explicitly
    non-strict exploratory runs.
    """

    def __init__(self, kind: str, seed: int = 42):
        self.kind = kind
        self.seed = seed
        self.transformer = None

    def fit(self, values: np.ndarray) -> FeatureScaler:
        if self.kind in {"fixed_log", "none"}:
            return self
        try:
            from sklearn.preprocessing import (
                MinMaxScaler,
                QuantileTransformer,
                RobustScaler,
                StandardScaler,
            )
        except ImportError as exc:
            raise RuntimeError("scikit-learn is required for feature scaling") from exc

        if self.kind == "quantile":
            self.transformer = QuantileTransformer(
                n_quantiles=min(1_000, len(values)),
                output_distribution="uniform",
                random_state=self.seed,
                subsample=min(200_000, len(values)),
            )
        elif self.kind == "robust":
            self.transformer = RobustScaler(quantile_range=(5.0, 95.0))
        elif self.kind == "standard":
            self.transformer = StandardScaler()
        elif self.kind == "minmax":
            self.transformer = MinMaxScaler(feature_range=(-1.0, 1.0))
        else:
            raise ValueError(f"Unknown scaler: {self.kind}")
        self.transformer.fit(values)
        return self

    def transform(self, values: np.ndarray) -> np.ndarray:
        if self.kind == "fixed_log":
            finite = np.nan_to_num(
                np.asarray(values, dtype=np.float64),
                nan=0.0,
                posinf=np.finfo(np.float64).max,
                neginf=-np.finfo(np.float64).max,
            )
            output = np.tanh(np.sign(finite) * np.log1p(np.abs(finite)) / 10.0)
        elif self.kind == "none":
            output = np.asarray(values)
        else:
            if self.transformer is None:
                raise RuntimeError("FeatureScaler must be fitted before transform")
            output = self.transformer.transform(values)
            if self.kind == "quantile":
                output = 2.0 * output - 1.0
            elif self.kind == "robust":
                output = np.clip(output / 4.0, -1.0, 1.0)
            elif self.kind == "minmax":
                output = np.clip(output, -1.0, 1.0)
        return np.nan_to_num(output, nan=0.0, posinf=0.0, neginf=0.0).astype(
            np.float32, copy=False
        )


def _fit_rows(arrays: Iterable[np.ndarray], limit: int | None, seed: int) -> np.ndarray:
    combined = np.concatenate(list(arrays), axis=0)
    if limit is None or len(combined) <= limit:
        return combined
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(combined), size=limit, replace=False)
    return combined[indices]


def _safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def _source_file_metadata(
    files: Sequence[Path], include_sha256: bool
) -> list[dict[str, object]]:
    output: list[dict[str, object]] = []
    for path in files:
        stat = path.stat()
        row: dict[str, object] = {
            "path": str(path),
            "size_bytes": int(stat.st_size),
            "modified_time_ns": int(stat.st_mtime_ns),
        }
        if include_sha256:
            row["sha256"] = _sha256_file(path)
        output.append(row)
    return output


def _sample_fingerprint(record_ids: np.ndarray, labels: np.ndarray) -> str:
    digest = hashlib.sha256()
    for record_id, label in zip(record_ids, labels, strict=True):
        digest.update(str(record_id).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(int(label)).encode("ascii"))
        digest.update(b"\n")
    return digest.hexdigest()


def prepare_data(
    config: DataConfig, output_dir: str | Path | None = None
) -> PreparedData:
    """Load all domains and produce reproducible, leakage-resistant splits."""

    config.validate()
    feature_frames: dict[str, pd.DataFrame] = {}
    labels_by_domain: dict[str, np.ndarray] = {}
    groups_by_domain: dict[str, np.ndarray] = {}
    record_ids_by_domain: dict[str, np.ndarray] = {}
    source_files: dict[str, list[str]] = {}
    rows_observed: dict[str, int] = {}
    source_metadata: dict[str, list[dict[str, object]]] = {}
    sample_fingerprints: dict[str, str] = {}
    duplicates_removed: dict[str, int] = {}
    requested = (
        [canonical_name(name) for name in config.common_features]
        if config.common_features is not None
        else None
    )

    for spec in config.datasets:
        files = resolve_dataset_files(spec, config.allow_kaggle_download)
        limit = spec.sample_rows
        if limit is None:
            limit = config.sample_rows_per_dataset
        raw = _uniform_sample_files(
            files,
            limit=limit,
            csv_chunk_rows=config.csv_chunk_rows,
            seed=_domain_seed(config.seed, spec.name, "row-sampling"),
            encoding=spec.encoding,
        )
        observed = int(raw.attrs.get("rows_observed", len(raw)))
        raw = _canonicalize_frame(raw)
        before_deduplication = len(raw)
        if config.drop_exact_duplicates:
            duplicate_columns = [
                column
                for column in raw.columns
                if column not in {"NF_SOURCE_FILE", "NF_SOURCE_ROW"}
            ]
            raw = raw.drop_duplicates(subset=duplicate_columns, keep="first")
            raw = raw.reset_index(drop=True)
        duplicates_removed[spec.name] = before_deduplication - len(raw)
        label_column = _find_label_column(raw, spec.label_column)
        labels, valid_mask = _binary_labels(
            raw[label_column],
            spec.name,
            require_zero_one_numeric=config.strict_protocol,
        )
        raw = raw.loc[valid_mask].reset_index(drop=True)
        if config.split_strategy == "group_stratified":
            groups = _make_group_ids(raw, config.group_columns)
        else:
            groups = np.array([f"row:{index}" for index in range(len(raw))])
        record_ids = (
            raw["NF_SOURCE_FILE"].astype(str) + "::" + raw["NF_SOURCE_ROW"].astype(str)
        ).to_numpy(dtype=str)
        features = _numeric_features(
            raw, label_column, spec.name, requested_features=requested
        )

        feature_frames[spec.name] = features
        labels_by_domain[spec.name] = labels
        groups_by_domain[spec.name] = groups
        record_ids_by_domain[spec.name] = record_ids
        source_files[spec.name] = [str(path) for path in files]
        rows_observed[spec.name] = observed
        source_metadata[spec.name] = _source_file_metadata(
            files, config.hash_source_files
        )
        sample_fingerprints[spec.name] = _sample_fingerprint(record_ids, labels)

    feature_sets = [set(frame.columns) for frame in feature_frames.values()]
    intersection = set.intersection(*feature_sets)
    if requested is not None:
        missing = {
            name: sorted(set(requested) - set(frame.columns))
            for name, frame in feature_frames.items()
        }
        missing = {name: columns for name, columns in missing.items() if columns}
        if missing:
            raise ValueError(f"Explicit common_features are missing: {missing}")
        common_features = requested
    else:
        # Preserve the first standardized dataset's feature ordering while
        # enforcing intersection membership across every domain.
        first = next(iter(feature_frames.values()))
        common_features = [column for column in first.columns if column in intersection]
    if len(common_features) < 2:
        by_domain = {
            name: list(frame.columns) for name, frame in feature_frames.items()
        }
        raise ValueError(
            "Fewer than two common numeric features remain across all datasets: "
            f"{by_domain}"
        )

    unscaled: dict[str, DomainSplit] = {}
    for spec in config.datasets:
        values = feature_frames[spec.name][common_features].to_numpy(
            dtype=np.float32, copy=True
        )
        labels = labels_by_domain[spec.name]
        groups = groups_by_domain[spec.name]
        record_ids = record_ids_by_domain[spec.name]
        if config.split_strategy == "group_stratified":
            train_idx, val_idx, test_idx = _stratified_group_indices(
                labels,
                groups,
                config.train_fraction,
                config.validation_fraction,
                seed=_domain_seed(config.seed, spec.name, "data-split"),
            )
        else:
            train_idx, val_idx, test_idx = _stratified_indices(
                labels,
                config.train_fraction,
                config.validation_fraction,
                seed=_domain_seed(config.seed, spec.name, "data-split"),
            )
        unscaled[spec.name] = DomainSplit(
            name=spec.name,
            x_train=values[train_idx],
            y_train=labels[train_idx],
            x_val=values[val_idx],
            y_val=labels[val_idx],
            x_test=values[test_idx],
            y_test=labels[test_idx],
            train_indices=train_idx,
            val_indices=val_idx,
            test_indices=test_idx,
            train_record_ids=record_ids[train_idx],
            val_record_ids=record_ids[val_idx],
            test_record_ids=record_ids[test_idx],
            train_group_ids=groups[train_idx],
            val_group_ids=groups[val_idx],
            test_group_ids=groups[test_idx],
        )

    if config.scaler in {"fixed_log", "none"}:
        # A dummy array makes the no-op fit API explicit without ever combining
        # or inspecting domain values.
        scaler_rows = np.empty((0, len(common_features)), dtype=np.float32)
    else:
        scaler_rows = _fit_rows(
            (split.x_train for split in unscaled.values()),
            config.scaler_fit_rows,
            config.seed,
        )
    scaler = FeatureScaler(config.scaler, config.seed).fit(scaler_rows)
    domains: dict[str, DomainSplit] = {}
    for name, split in unscaled.items():
        domains[name] = DomainSplit(
            name=name,
            x_train=scaler.transform(split.x_train),
            y_train=split.y_train,
            x_val=scaler.transform(split.x_val),
            y_val=split.y_val,
            x_test=scaler.transform(split.x_test),
            y_test=split.y_test,
            train_indices=split.train_indices,
            val_indices=split.val_indices,
            test_indices=split.test_indices,
            train_record_ids=split.train_record_ids,
            val_record_ids=split.val_record_ids,
            test_record_ids=split.test_record_ids,
            train_group_ids=split.train_group_ids,
            val_group_ids=split.val_group_ids,
            test_group_ids=split.test_group_ids,
        )

    prepared = PreparedData(
        domains=domains,
        common_features=common_features,
        scaler=scaler,
        source_files=source_files,
        rows_observed=rows_observed,
        source_metadata=source_metadata,
        sample_fingerprints=sample_fingerprints,
        duplicates_removed=duplicates_removed,
    )

    if output_dir is not None:
        destination = Path(output_dir)
        destination.mkdir(parents=True, exist_ok=True)
        feature_payload = {
            "feature_count": len(common_features),
            "features": common_features,
            "domain_feature_counts_before_intersection": {
                name: len(frame.columns) for name, frame in feature_frames.items()
            },
            "dropped_noncommon_features": {
                name: [
                    column for column in frame.columns if column not in common_features
                ]
                for name, frame in feature_frames.items()
            },
            "scaler": config.scaler,
            "feature_schema_scope": (
                "predeclared" if requested is not None else "observed_intersection"
            ),
            "transform_scope": (
                "stateless_data_independent"
                if config.scaler in {"fixed_log", "none"}
                else "fitted_combined_training_partitions_exploratory_only"
            ),
            "claim_scope": (
                "full_configured_model_pipeline"
                if config.scaler in {"fixed_log", "none"} and requested is not None
                else "model_weights_only"
            ),
        }
        (destination / "common_features.json").write_text(
            json.dumps(feature_payload, indent=2), encoding="utf-8"
        )

        manifest: dict[str, object] = {
            "seed": config.seed,
            "fractions": {
                "train": config.train_fraction,
                "validation": config.validation_fraction,
                "test": config.test_fraction,
            },
            "split_strategy": config.split_strategy,
            "group_columns": config.group_columns,
            "drop_exact_duplicates": config.drop_exact_duplicates,
            "domains": {},
        }
        domain_manifest = manifest["domains"]
        assert isinstance(domain_manifest, dict)
        for name, split in domains.items():
            domain_manifest[name] = {
                "rows_observed_before_sampling": rows_observed[name],
                "duplicates_removed_after_sampling": duplicates_removed[name],
                "rows_after_sampling": int(
                    len(split.y_train) + len(split.y_val) + len(split.y_test)
                ),
                "source_files": source_files[name],
                "source_file_metadata": source_metadata[name],
                "sample_fingerprint_sha256": sample_fingerprints[name],
                "splits": {
                    "train": {
                        "rows": len(split.y_train),
                        "attack_rate": float(split.y_train.mean()),
                    },
                    "validation": {
                        "rows": len(split.y_val),
                        "attack_rate": float(split.y_val.mean()),
                    },
                    "test": {
                        "rows": len(split.y_test),
                        "attack_rate": float(split.y_test.mean()),
                    },
                },
            }
            np.savez_compressed(
                destination / f"{_safe_name(name)}_split_indices.npz",
                train=split.train_indices,
                validation=split.val_indices,
                test=split.test_indices,
                train_labels=split.y_train,
                validation_labels=split.y_val,
                test_labels=split.y_test,
                train_record_ids=split.train_record_ids,
                validation_record_ids=split.val_record_ids,
                test_record_ids=split.test_record_ids,
                train_group_ids=split.train_group_ids,
                validation_group_ids=split.val_group_ids,
                test_group_ids=split.test_group_ids,
            )
        (destination / "split_manifest.json").write_text(
            json.dumps(manifest, indent=2), encoding="utf-8"
        )
        try:
            import joblib

            # Saving the wrapper class directly is fragile when this code is
            # executed from a notebook because its module is ``__main__``.
            # The sklearn transformer itself is importable and portable.
            joblib.dump(
                {
                    "kind": scaler.kind,
                    "seed": scaler.seed,
                    "formula": (
                        "tanh(sign(x) * log1p(abs(x)) / 10)"
                        if scaler.kind == "fixed_log"
                        else None
                    ),
                    "transformer": scaler.transformer,
                },
                destination / "scaler.joblib",
            )
        except ImportError as exc:
            raise RuntimeError(
                "joblib (installed with scikit-learn) is required"
            ) from exc

    return prepared


## 4. MLP and numerical-feature Transformer

**What the following block does:** This cell defines two classifiers with the same input and output contract. `MLPClassifier` passes each row through linear layers, LayerNorm, GELU activations, and dropout. The second model gives every scalar feature its own learned token projection, applies Transformer self-attention across feature tokens, mean-pools the tokens, and classifies the result. This is an FT-style numerical feature Transformer rather than the original categorical-only TabTransformer, although `tabtransformer` is retained as the configuration name requested for the project. Both networks output two raw logits—one for benign and one for attack—and later training uses softmax-compatible cross-entropy. The helper functions construct a model and report its parameter and checkpoint sizes; this cell performs no optimization.

In [10]:
"""MLP and FT-style numerical-feature Transformer classifiers."""

from __future__ import annotations

import torch
from torch import nn



class MLPClassifier(nn.Module):
    """Explicit MLP replacement for the notebook's kernel-one CNN baseline."""

    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        latent_dim: int,
        dropout: float,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        previous = input_dim
        for width in hidden_dims:
            layers.extend(
                [
                    nn.Linear(previous, width),
                    nn.LayerNorm(width),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            previous = width
        layers.extend(
            [
                nn.Linear(previous, latent_dim),
                nn.LayerNorm(latent_dim),
                nn.GELU(),
            ]
        )
        self.encoder = nn.Sequential(*layers)
        self.classifier = nn.Linear(latent_dim, 2)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward_features(self, values: torch.Tensor) -> torch.Tensor:
        return self.encoder(values)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.forward_features(values))


class NumericalFeatureTokenizer(nn.Module):
    """Turn each scalar continuous feature into its own learned token."""

    def __init__(self, input_dim: int, d_token: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, d_token))
        self.bias = nn.Parameter(torch.empty(input_dim, d_token))
        nn.init.normal_(self.weight, std=0.02)
        nn.init.normal_(self.bias, std=0.02)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return values.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class TabTransformerClassifier(nn.Module):
    """FT-style continuous-token Transformer matching the requested setup.

    Standard NetFlow common features are numerical, so categorical embeddings
    are unnecessary. Each scalar gets a learned feature-specific projection,
    self-attention models cross-feature interactions, and mean pooling yields a
    fixed-size representation. This is not the original categorical
    TabTransformer; the class keeps that project-facing name for compatibility.
    """

    def __init__(
        self,
        input_dim: int,
        latent_dim: int,
        d_token: int,
        n_heads: int,
        n_layers: int,
        ffn_factor: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.tokenizer = NumericalFeatureTokenizer(input_dim, d_token)
        layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ffn_factor,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers=n_layers, enable_nested_tensor=False
        )
        self.final_token_norm = nn.LayerNorm(d_token)
        self.encoder_out = nn.Sequential(
            nn.Linear(d_token, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.GELU(),
        )
        self.classifier = nn.Linear(latent_dim, 2)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        # TransformerEncoderLayer has already initialized its parameters.  Use
        # the same stable initialization as the MLP for the added projections.
        for module in (self.encoder_out[0], self.classifier):
            assert isinstance(module, nn.Linear)
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward_features(self, values: torch.Tensor) -> torch.Tensor:
        tokens = self.transformer(self.tokenizer(values))
        pooled = self.final_token_norm(tokens).mean(dim=1)
        return self.encoder_out(pooled)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.forward_features(values))


def build_model(name: str, input_dim: int, config: ModelConfig) -> nn.Module:
    normalized = name.strip().lower()
    if normalized == "mlp":
        return MLPClassifier(
            input_dim=input_dim,
            hidden_dims=config.mlp_hidden_dims,
            latent_dim=config.latent_dim,
            dropout=config.dropout,
        )
    if normalized == "tabtransformer":
        return TabTransformerClassifier(
            input_dim=input_dim,
            latent_dim=config.latent_dim,
            d_token=config.tab_d_token,
            n_heads=config.tab_heads,
            n_layers=config.tab_layers,
            ffn_factor=config.tab_ffn_factor,
            dropout=config.dropout,
        )
    raise ValueError(f"Unknown architecture {name!r}; choose mlp or tabtransformer")


def parameter_count(model: nn.Module, trainable_only: bool = False) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if not trainable_only or parameter.requires_grad
    )


def state_nbytes(state: dict[str, torch.Tensor]) -> int:
    return sum(tensor.numel() * tensor.element_size() for tensor in state.values())


## 5. Pooled multi-domain training and update logging

**What the following block does:** This cell defines reproducible training. It creates domain-homogeneous mini-batches so every optimization step can be attributed to exactly one dataset. Row permutations, schedule priorities, and dropout seeds are derived independently for each domain and batch; filtering a forgotten domain therefore leaves retained batches in the same order with the same examples and dropout masks in the scratch counterfactual. Strict runs use ordinary SGD with zero momentum and zero weight decay plus fixed, predeclared cross-entropy weights. This avoids AdamW moment buffers, regularization terms, and data-derived class weights that cannot be cleanly assigned to one domain.

During original training, the code stores the literal parameter difference from every step in that batch's domain trace. Initialization plus all traces must reconstruct the final checkpoint, which is checked later. A fixed final epoch prevents a future forgotten domain from influencing checkpoint choice through early stopping. Validation is run once at the end in strict mode, and optimization time is recorded separately from validation time so the efficiency comparison is like-for-like. The cell also defines prediction and checkpoint-copying helpers; actual training starts in Section 12.

In [11]:
"""Deterministic multi-domain training with per-domain update accounting."""

from __future__ import annotations

import hashlib
import math
import random
import time
from collections.abc import Iterator, Mapping, Sequence
from dataclasses import dataclass

import numpy as np
import torch
from torch import nn


TensorState = dict[str, torch.Tensor]
DomainDeltas = dict[str, TensorState]


def seed_everything(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:  # older torch
            torch.use_deterministic_algorithms(True)
        if torch.backends.cudnn.is_available():
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True


def resolve_device(requested: str) -> torch.device:
    if requested == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        mps = getattr(torch.backends, "mps", None)
        if mps is not None and mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    device = torch.device(requested)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    if device.type == "mps":
        mps = getattr(torch.backends, "mps", None)
        if mps is None or not mps.is_available():
            raise RuntimeError("MPS was requested but is unavailable")
    return device


def synchronize(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def cpu_state_dict(model: nn.Module) -> TensorState:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def _cpu_deltas(deltas: DomainDeltas) -> DomainDeltas:
    return {
        domain: {name: tensor.detach().cpu().clone() for name, tensor in state.items()}
        for domain, state in deltas.items()
    }


def load_state_dict(model: nn.Module, state: Mapping[str, torch.Tensor]) -> None:
    model.load_state_dict({name: tensor.clone() for name, tensor in state.items()})


def fixed_class_weights(config: TrainingConfig, device: torch.device) -> torch.Tensor:
    """Return predeclared loss weights that no dataset can influence."""

    return torch.tensor(config.fixed_class_weights, dtype=torch.float32, device=device)


def make_optimizer(
    parameters: Iterator[nn.Parameter] | Sequence[nn.Parameter],
    config: TrainingConfig,
    learning_rate: float | None = None,
) -> torch.optim.Optimizer:
    """Construct the configured optimizer and keep strict SGD explicit."""

    lr = config.learning_rate if learning_rate is None else learning_rate
    if config.optimizer == "sgd":
        return torch.optim.SGD(
            parameters,
            lr=lr,
            momentum=config.sgd_momentum,
            weight_decay=config.weight_decay,
        )
    if config.optimizer == "adamw":
        return torch.optim.AdamW(parameters, lr=lr, weight_decay=config.weight_decay)
    raise ValueError(f"Unknown optimizer: {config.optimizer}")


def _stable_seed(*parts: object) -> int:
    payload = "\x1f".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little")


def _stratified_fraction_indices(
    labels: np.ndarray, fraction: float, rng: np.random.Generator
) -> np.ndarray:
    if fraction >= 1.0:
        indices = np.arange(len(labels), dtype=np.int64)
        rng.shuffle(indices)
        return indices
    parts: list[np.ndarray] = []
    for label in (0, 1):
        candidates = np.flatnonzero(labels == label)
        count = max(1, round(len(candidates) * fraction))
        parts.append(
            rng.choice(candidates, size=min(count, len(candidates)), replace=False)
        )
    indices = np.concatenate(parts).astype(np.int64, copy=False)
    rng.shuffle(indices)
    return indices


def iter_domain_batches(
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    batch_size: int,
    seed: int,
    sampling: str = "proportional",
    fraction: float = 1.0,
) -> Iterator[tuple[str, np.ndarray, np.ndarray, int]]:
    """Yield domain-homogeneous batches in a shuffled multi-domain schedule.

    Homogeneous batches are required for attributing each optimizer update to a
    domain.  ``proportional`` traverses every selected row once.  ``balanced``
    cycles smaller domains until each domain contributes the same batch count.
    """

    chunks: dict[str, list[np.ndarray]] = {}
    for name in domain_names:
        rng = np.random.default_rng(_stable_seed(seed, "rows", name, fraction))
        chosen = _stratified_fraction_indices(domains[name].y_train, fraction, rng)
        chunks[name] = [
            chosen[start : start + batch_size]
            for start in range(0, len(chosen), batch_size)
        ]

    schedule: list[tuple[str, int]] = []
    if sampling == "proportional":
        for name in domain_names:
            schedule.extend((name, batch) for batch in range(len(chunks[name])))
    elif sampling == "balanced":
        max_batches = max(len(chunks[name]) for name in domain_names)
        for name in domain_names:
            for batch in range(max_batches):
                schedule.append((name, batch % len(chunks[name])))
    else:
        raise ValueError(f"Unknown domain sampling mode: {sampling}")
    # Each batch receives a data-independent priority.  Filtering one domain
    # from a full schedule therefore leaves the retained batches in exactly the
    # same relative order as a leave-one-domain-out scratch run.
    schedule.sort(
        key=lambda item: (
            _stable_seed(seed, "schedule", item[0], item[1]),
            item[0],
            item[1],
        )
    )

    for name, batch_number in schedule:
        selected = chunks[name][batch_number]
        batch_seed = _stable_seed(seed, "batch", name, batch_number) % (2**63 - 1)
        yield (
            name,
            domains[name].x_train[selected],
            domains[name].y_train[selected],
            batch_seed,
        )


@dataclass
class TrainingResult:
    state_dict: TensorState
    domain_deltas: DomainDeltas | None
    history: list[dict[str, float]]
    elapsed_seconds: float
    optimization_seconds: float
    validation_seconds: float
    validation_evaluations: int
    optimizer_steps: int
    samples_seen: int
    best_epoch: int
    best_validation_loss: float
    checkpoint_bytes: int
    delta_bytes: int


@torch.inference_mode()
def validation_loss(
    model: nn.Module,
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    criterion: nn.Module,
    device: torch.device,
    batch_size: int,
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    count = 0
    for name in domain_names:
        values = domains[name].x_val
        labels = domains[name].y_val
        for start in range(0, len(labels), batch_size):
            x = torch.from_numpy(values[start : start + batch_size]).to(device)
            y = torch.from_numpy(labels[start : start + batch_size]).long().to(device)
            logits = model(x)
            loss = criterion(logits, y)
            batch_count = len(y)
            total_loss += float(loss.item()) * batch_count
            correct += int((logits.argmax(dim=1) == y).sum().item())
            count += batch_count
    return total_loss / max(count, 1), correct / max(count, 1)


def train_model(
    model: nn.Module,
    domains: Mapping[str, DomainSplit],
    config: TrainingConfig,
    device: torch.device,
    seed: int,
    domain_names: Sequence[str] | None = None,
    track_domain_deltas: bool = False,
    deterministic: bool = True,
) -> TrainingResult:
    """Train on the union of selected domains and retain the selected checkpoint.

    When tracking is enabled, every pure-domain SGD parameter delta is added
    to that domain's cumulative buffer.  At any checkpoint the model parameters
    equal initialization plus the sum of these buffers (up to floating-point
    accumulation).  Subtracting one buffer is the amnesiac approximation used
    by :mod:`netflow_unlearning.unlearning`.
    """

    selected = list(domain_names or domains.keys())
    if not selected:
        raise ValueError("At least one training domain is required")
    seed_everything(seed, deterministic)
    model.to(device)
    optimizer = make_optimizer(model.parameters(), config)
    criterion = nn.CrossEntropyLoss(weight=fixed_class_weights(config, device))

    parameter_map = dict(model.named_parameters())
    deltas: DomainDeltas | None = None
    if track_domain_deltas:
        deltas = {
            domain: {
                name: torch.zeros_like(parameter, device=device)
                for name, parameter in parameter_map.items()
            }
            for domain in selected
        }

    best_state: TensorState | None = None
    best_deltas: DomainDeltas | None = None
    best_loss = math.inf
    best_epoch = 0
    stale_epochs = 0
    steps = 0
    samples = 0
    history: list[dict[str, float]] = []
    optimization_seconds = 0.0
    validation_seconds = 0.0
    validation_evaluations = 0

    synchronize(device)
    started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_samples = 0
        epoch_optimization_started = time.perf_counter()
        for domain, x_np, y_np, batch_seed in iter_domain_batches(
            domains,
            selected,
            batch_size=config.batch_size,
            seed=seed + epoch * 1_000_003,
            sampling=config.domain_sampling,
        ):
            # A stable per-batch seed pairs dropout masks between the original
            # and retained-only counterfactual schedules.
            torch.manual_seed(batch_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(batch_seed)
            x = torch.from_numpy(x_np).to(device)
            y = torch.from_numpy(y_np).long().to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            if config.gradient_clip_norm > 0:
                nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)

            before: dict[str, torch.Tensor] | None = None
            if deltas is not None:
                before = {
                    name: parameter.detach().clone()
                    for name, parameter in parameter_map.items()
                }
            optimizer.step()
            if deltas is not None and before is not None:
                with torch.no_grad():
                    for name, parameter in parameter_map.items():
                        deltas[domain][name].add_(parameter.detach() - before[name])

            batch_count = len(y)
            epoch_loss += float(loss.item()) * batch_count
            epoch_samples += batch_count
            steps += 1
            samples += batch_count
        synchronize(device)
        optimization_seconds += time.perf_counter() - epoch_optimization_started

        should_validate = (
            config.checkpoint_selection == "early_stopping" or epoch == config.epochs
        )
        if should_validate:
            validation_started = time.perf_counter()
            val_loss, val_accuracy = validation_loss(
                model, domains, selected, criterion, device, config.batch_size
            )
            synchronize(device)
            validation_seconds += time.perf_counter() - validation_started
            validation_evaluations += 1
        else:
            val_loss, val_accuracy = math.nan, math.nan
        history.append(
            {
                "epoch": float(epoch),
                "training_loss": epoch_loss / max(epoch_samples, 1),
                "validation_loss": val_loss,
                "validation_accuracy": val_accuracy,
            }
        )
        if config.checkpoint_selection == "final":
            # Selecting the original epoch with a validation domain that is
            # later forgotten would itself retain that domain's influence.  A
            # fixed epoch budget avoids this non-parameter deletion channel.
            if epoch == config.epochs:
                best_loss = val_loss
                best_epoch = epoch
                best_state = cpu_state_dict(model)
                best_deltas = _cpu_deltas(deltas) if deltas is not None else None
        elif val_loss < best_loss - config.min_delta:
            best_loss = val_loss
            best_epoch = epoch
            best_state = cpu_state_dict(model)
            best_deltas = _cpu_deltas(deltas) if deltas is not None else None
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= config.patience:
                break

    synchronize(device)
    elapsed = time.perf_counter() - started
    if best_state is None:
        raise RuntimeError("Training completed without producing a checkpoint")
    load_state_dict(model, best_state)
    delta_bytes = 0
    if best_deltas is not None:
        delta_bytes = sum(state_nbytes(state) for state in best_deltas.values())
    return TrainingResult(
        state_dict=best_state,
        domain_deltas=best_deltas,
        history=history,
        elapsed_seconds=elapsed,
        optimization_seconds=optimization_seconds,
        validation_seconds=validation_seconds,
        validation_evaluations=validation_evaluations,
        optimizer_steps=steps,
        samples_seen=samples,
        best_epoch=best_epoch,
        best_validation_loss=best_loss,
        checkpoint_bytes=state_nbytes(best_state),
        delta_bytes=delta_bytes,
    )


@torch.inference_mode()
def predict_proba(
    model: nn.Module,
    values: np.ndarray,
    device: torch.device,
    batch_size: int = 1_024,
) -> np.ndarray:
    model.eval()
    model.to(device)
    outputs: list[np.ndarray] = []
    for start in range(0, len(values), batch_size):
        x = torch.from_numpy(values[start : start + batch_size]).to(device)
        probabilities = torch.softmax(model(x), dim=1)
        outputs.append(probabilities.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)


## 6. Approximate domain unlearning

**What the following block does:** This cell implements amnesiac update rollback with retained-data repair. For a requested domain, `rollback_state` subtracts that domain's accumulated SGD parameter changes from the original checkpoint. This exactly removes the logged additive changes themselves, but it cannot undo interaction effects: later retained updates were evaluated at weights already altered by the forgotten domain. `unlearn_domain` therefore fine-tunes the rolled-back state for a small number of steps using only a configured fraction of retained training rows. It never uses the forgotten domain, validation partitions, or test partitions during repair. Each deletion starts independently from the same original checkpoint, and the result records rollback time, repair time, processed samples, optimizer steps, and rollback magnitude. The method is explicitly approximate; closeness to scratch retraining must be established by the later measurements rather than assumed.

In [12]:
"""Architecture-neutral amnesiac domain rollback with retained-data repair."""

from __future__ import annotations

import time
from collections.abc import Mapping
from dataclasses import dataclass

import numpy as np
import torch
from torch import nn



@dataclass
class UnlearningResult:
    state_dict: TensorState
    elapsed_seconds: float
    rollback_seconds: float
    repair_optimization_seconds: float
    optimizer_steps: int
    samples_seen: int
    rollback_l2: float
    rollback_max_abs: float


def rollback_state(
    original_state: Mapping[str, torch.Tensor],
    forget_delta: Mapping[str, torch.Tensor],
    scale: float = 1.0,
) -> TensorState:
    """Subtract the cumulative updates attributed to the forgotten domain."""

    rolled: TensorState = {}
    for name, value in original_state.items():
        result = value.detach().cpu().clone()
        if name in forget_delta:
            delta = forget_delta[name].detach().cpu().to(result.dtype)
            if delta.shape != result.shape:
                raise ValueError(
                    f"Delta shape mismatch for {name}: {delta.shape} vs {result.shape}"
                )
            result.sub_(delta, alpha=scale)
        rolled[name] = result
    extra = set(forget_delta) - set(original_state)
    if extra:
        raise ValueError(f"Forget delta contains unknown parameters: {sorted(extra)}")
    return rolled


def unlearn_domain(
    model: nn.Module,
    original_state: Mapping[str, torch.Tensor],
    domain_deltas: DomainDeltas,
    forget_domain: str,
    domains: Mapping[str, DomainSplit],
    training_config: TrainingConfig,
    unlearning_config: UnlearningConfig,
    device: torch.device,
    seed: int,
) -> UnlearningResult:
    """Independently forget one domain from the shared original checkpoint.

    Phase 1 removes that domain's logged optimizer trajectory.  Because later
    retained-domain updates were computed at parameters influenced by the
    forgotten updates, subtraction is approximate rather than exact retraining.
    Phase 2 repairs those interaction/order errors with a deliberately small,
    training-only subset of retained domains.  No forgotten, validation, or test
    record is used by the repair phase.
    """

    if forget_domain not in domain_deltas:
        raise KeyError(f"No logged updates for forgotten domain {forget_domain!r}")
    retained = [name for name in domains if name != forget_domain]
    if not retained:
        raise ValueError("Cannot forget the only domain")

    seed_everything(seed)
    synchronize(device)
    started = time.perf_counter()
    rollback_started = started
    rolled = rollback_state(
        original_state,
        domain_deltas[forget_domain],
        scale=unlearning_config.rollback_scale,
    )
    load_state_dict(model, rolled)
    model.to(device)
    synchronize(device)
    rollback_seconds = time.perf_counter() - rollback_started

    squared = 0.0
    max_abs = 0.0
    for name, value in original_state.items():
        difference = rolled[name].float() - value.detach().cpu().float()
        squared += float(torch.sum(difference * difference).item())
        if difference.numel():
            max_abs = max(max_abs, float(difference.abs().max().item()))

    steps = 0
    samples = 0
    repair_seconds = 0.0
    if unlearning_config.repair_epochs > 0:
        optimizer = make_optimizer(
            model.parameters(),
            training_config,
            learning_rate=unlearning_config.repair_learning_rate,
        )
        criterion = nn.CrossEntropyLoss(
            weight=fixed_class_weights(training_config, device)
        )
        repair_started = time.perf_counter()
        for epoch in range(1, unlearning_config.repair_epochs + 1):
            model.train()
            for _, x_np, y_np, batch_seed in iter_domain_batches(
                domains,
                retained,
                batch_size=training_config.batch_size,
                seed=seed + epoch * 1_000_033,
                sampling=training_config.domain_sampling,
                fraction=unlearning_config.repair_fraction,
            ):
                torch.manual_seed(batch_seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(batch_seed)
                x = torch.from_numpy(x_np).to(device)
                y = torch.from_numpy(y_np).long().to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(x), y)
                loss.backward()
                if training_config.gradient_clip_norm > 0:
                    nn.utils.clip_grad_norm_(
                        model.parameters(), training_config.gradient_clip_norm
                    )
                optimizer.step()
                steps += 1
                samples += len(y)
        synchronize(device)
        repair_seconds = time.perf_counter() - repair_started

    synchronize(device)
    elapsed = time.perf_counter() - started
    return UnlearningResult(
        state_dict=cpu_state_dict(model),
        elapsed_seconds=elapsed,
        rollback_seconds=rollback_seconds,
        repair_optimization_seconds=repair_seconds,
        optimizer_steps=steps,
        samples_seen=samples,
        rollback_l2=float(np.sqrt(squared)),
        rollback_max_abs=max_abs,
    )


## 7. Utility, counterfactual similarity, and privacy audits

**What the following block does:** This cell defines three evaluation families. `classification_metrics` computes accuracy, balanced accuracy, attack precision/recall/F1, ROC AUC, average precision, log loss, and the confusion matrix. `prediction_similarity` directly compares a candidate with the scratch reference on identical records using predicted-label agreement, mean absolute and root-mean-square attack-probability gaps, and Jensen–Shannon divergence. These comparisons matter because forgetting means approaching the no-domain counterfactual, not forcing forgotten-domain accuracy to chance.

For privacy, the cell selects forgotten-domain training records as putative members and disjoint test records as nonmembers, with identical benign/attack counts; the exact same sampled indices are reused for all three models. It reports a predetermined loss-threshold attack with AUC, advantage, TPR at fixed FPR, and stratified bootstrap confidence intervals; candidate-minus-scratch MIA AUC gaps receive paired bootstrap intervals using the same resampled records. It also trains a logistic output attack on one half of the sampled audit records and evaluates it on the disjoint half using loss, confidence, entropy, and margin. These attacks can reveal residual membership signal, but failing to detect signal is not a mathematical or differential-privacy guarantee.

In [13]:
"""Utility metrics, gold-model similarity, and a same-domain MIA."""

from __future__ import annotations

import numpy as np

EPSILON = 1e-12


def classification_metrics(
    y_true: np.ndarray, probabilities: np.ndarray
) -> dict[str, float]:
    """Return binary intrusion-detection metrics with attack as positive class."""

    try:
        from sklearn.metrics import (
            accuracy_score,
            average_precision_score,
            balanced_accuracy_score,
            confusion_matrix,
            f1_score,
            log_loss,
            precision_score,
            recall_score,
            roc_auc_score,
        )
    except ImportError as exc:
        raise RuntimeError("scikit-learn is required for evaluation") from exc

    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != 2:
        raise ValueError(
            f"Expected N x 2 probabilities, received {probabilities.shape}"
        )
    if len(y_true) != len(probabilities):
        raise ValueError("Labels and probabilities have different lengths")
    probabilities = np.clip(probabilities, EPSILON, 1.0)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    positive = np.clip(probabilities[:, 1], EPSILON, 1.0 - EPSILON)
    predictions = (positive >= 0.5).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "precision_attack": float(
            precision_score(y_true, predictions, pos_label=1, zero_division=0)
        ),
        "recall_attack": float(
            recall_score(y_true, predictions, pos_label=1, zero_division=0)
        ),
        "f1_attack": float(f1_score(y_true, predictions, pos_label=1, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, positive)),
        "average_precision": float(average_precision_score(y_true, positive)),
        "log_loss": float(log_loss(y_true, probabilities, labels=[0, 1])),
        "true_negative": float(tn),
        "false_positive": float(fp),
        "false_negative": float(fn),
        "true_positive": float(tp),
        "samples": float(len(y_true)),
        "attack_rate": float(np.mean(y_true)),
    }


def prediction_similarity(candidate: np.ndarray, gold: np.ndarray) -> dict[str, float]:
    """Measure functional closeness to a scratch leave-one-domain-out model."""

    candidate = np.clip(np.asarray(candidate, dtype=np.float64), EPSILON, 1.0)
    gold = np.clip(np.asarray(gold, dtype=np.float64), EPSILON, 1.0)
    if candidate.shape != gold.shape:
        raise ValueError(f"Prediction shapes differ: {candidate.shape} vs {gold.shape}")
    candidate /= candidate.sum(axis=1, keepdims=True)
    gold /= gold.sum(axis=1, keepdims=True)
    midpoint = 0.5 * (candidate + gold)
    js_per_row = 0.5 * np.sum(candidate * np.log(candidate / midpoint), axis=1)
    js_per_row += 0.5 * np.sum(gold * np.log(gold / midpoint), axis=1)
    return {
        "prediction_agreement": float(
            np.mean(candidate.argmax(axis=1) == gold.argmax(axis=1))
        ),
        "mean_abs_attack_probability_gap": float(
            np.mean(np.abs(candidate[:, 1] - gold[:, 1]))
        ),
        "root_mean_square_probability_gap": float(
            np.sqrt(np.mean((candidate[:, 1] - gold[:, 1]) ** 2))
        ),
        "mean_jensen_shannon_divergence": float(np.mean(js_per_row)),
    }


def matched_membership_indices(
    member_labels: np.ndarray,
    nonmember_labels: np.ndarray,
    max_samples_per_class: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, dict[int, int]]:
    rng = np.random.default_rng(seed)
    member_parts = []
    nonmember_parts = []
    counts: dict[int, int] = {}
    for label in (0, 1):
        members = np.flatnonzero(member_labels == label)
        nonmembers = np.flatnonzero(nonmember_labels == label)
        count = min(len(members), len(nonmembers), max_samples_per_class)
        if count < 2:
            raise ValueError(
                f"Membership attack needs >=2 member/nonmember rows for class {label}"
            )
        member_parts.append(rng.choice(members, size=count, replace=False))
        nonmember_parts.append(rng.choice(nonmembers, size=count, replace=False))
        counts[label] = count
    member_indices = np.concatenate(member_parts)
    nonmember_indices = np.concatenate(nonmember_parts)
    rng.shuffle(member_indices)
    rng.shuffle(nonmember_indices)
    return member_indices, nonmember_indices, counts


# Backward-compatible private spelling used by early notebook versions.
_matched_membership_indices = matched_membership_indices


def _loss_attack_statistics(
    member_loss: np.ndarray, nonmember_loss: np.ndarray, fixed_fpr: float
) -> dict[str, float]:
    from sklearn.metrics import roc_auc_score, roc_curve

    membership = np.concatenate(
        [
            np.ones(len(member_loss), dtype=np.int64),
            np.zeros(len(nonmember_loss), dtype=np.int64),
        ]
    )
    score = np.concatenate([-member_loss, -nonmember_loss])
    fpr, tpr, _ = roc_curve(membership, score)
    valid = tpr[fpr <= fixed_fpr]
    return {
        "mia_loss_auc": float(roc_auc_score(membership, score)),
        "mia_advantage": float(np.max(tpr - fpr)),
        "mia_tpr_at_fixed_fpr": float(valid.max()) if len(valid) else 0.0,
        "member_nonmember_loss_gap": float(nonmember_loss.mean() - member_loss.mean()),
    }


def _learned_output_attack(
    member_labels: np.ndarray,
    member_probabilities: np.ndarray,
    nonmember_labels: np.ndarray,
    nonmember_probabilities: np.ndarray,
    fixed_fpr: float,
    seed: int,
) -> dict[str, float]:
    """Fit/evaluate a disjoint logistic output attack as a sensitivity check."""

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, roc_curve
    from sklearn.model_selection import train_test_split

    labels = np.concatenate([member_labels, nonmember_labels]).astype(np.int64)
    probabilities = np.concatenate(
        [member_probabilities, nonmember_probabilities], axis=0
    )
    membership = np.concatenate(
        [
            np.ones(len(member_labels), dtype=np.int64),
            np.zeros(len(nonmember_labels), dtype=np.int64),
        ]
    )
    true_confidence = probabilities[np.arange(len(labels)), labels]
    other_confidence = probabilities[np.arange(len(labels)), 1 - labels]
    loss = -np.log(np.clip(true_confidence, EPSILON, 1.0))
    entropy = -np.sum(
        probabilities * np.log(np.clip(probabilities, EPSILON, 1.0)), axis=1
    )
    margin = true_confidence - other_confidence
    features = np.column_stack([loss, true_confidence, entropy, margin])
    strata = 2 * membership + labels
    train_idx, test_idx = train_test_split(
        np.arange(len(labels)),
        test_size=0.5,
        random_state=seed,
        stratify=strata,
    )
    attack = LogisticRegression(
        max_iter=1_000, class_weight="balanced", random_state=seed
    )
    attack.fit(features[train_idx], membership[train_idx])
    score = attack.predict_proba(features[test_idx])[:, 1]
    truth = membership[test_idx]
    fpr, tpr, _ = roc_curve(truth, score)
    valid = tpr[fpr <= fixed_fpr]
    return {
        "mia_learned_auc": float(roc_auc_score(truth, score)),
        "mia_learned_advantage": float(np.max(tpr - fpr)),
        "mia_learned_tpr_at_fixed_fpr": (float(valid.max()) if len(valid) else 0.0),
        "mia_learned_attack_train_rows": float(len(train_idx)),
        "mia_learned_attack_test_rows": float(len(test_idx)),
    }


def loss_membership_attack(
    member_labels: np.ndarray,
    member_probabilities: np.ndarray,
    nonmember_labels: np.ndarray,
    nonmember_probabilities: np.ndarray,
    max_samples_per_class: int,
    fixed_fpr: float,
    seed: int,
    bootstrap_repetitions: int = 1_000,
    confidence_level: float = 0.95,
    member_indices: np.ndarray | None = None,
    nonmember_indices: np.ndarray | None = None,
) -> dict[str, float]:
    """Run a predetermined loss-threshold membership inference attack.

    Members are forgotten-domain *training* records and nonmembers are its
    disjoint test records.  Sampling matches the class counts between the two
    groups so the attack cannot exploit benign/attack prevalence.  Lower target
    loss is the fixed membership score direction; no target test labels are used
    to fit a separate attack classifier.
    """

    try:
        import sklearn  # noqa: F401
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn is required for the membership attack"
        ) from exc

    if (member_indices is None) != (nonmember_indices is None):
        raise ValueError("Provide both matched index arrays or neither")
    if member_indices is None:
        member_idx, nonmember_idx, counts = matched_membership_indices(
            member_labels, nonmember_labels, max_samples_per_class, seed
        )
    else:
        member_idx = np.asarray(member_indices, dtype=np.int64)
        nonmember_idx = np.asarray(nonmember_indices, dtype=np.int64)
        if len(member_idx) != len(nonmember_idx):
            raise ValueError("Matched membership groups must have equal size")
        counts = {
            label: int(np.sum(member_labels[member_idx] == label)) for label in (0, 1)
        }
        if any(
            counts[label] != int(np.sum(nonmember_labels[nonmember_idx] == label))
            for label in (0, 1)
        ):
            raise ValueError("Provided membership indices are not class matched")
    member_probs = np.clip(member_probabilities[member_idx], EPSILON, 1.0)
    nonmember_probs = np.clip(nonmember_probabilities[nonmember_idx], EPSILON, 1.0)
    member_loss = -np.log(
        member_probs[np.arange(len(member_idx)), member_labels[member_idx]]
    )
    nonmember_loss = -np.log(
        nonmember_probs[np.arange(len(nonmember_idx)), nonmember_labels[nonmember_idx]]
    )
    result = {
        **_loss_attack_statistics(member_loss, nonmember_loss, fixed_fpr),
        "fixed_fpr": float(fixed_fpr),
        "member_mean_loss": float(member_loss.mean()),
        "nonmember_mean_loss": float(nonmember_loss.mean()),
        "member_nonmember_loss_gap": float(nonmember_loss.mean() - member_loss.mean()),
        "samples_per_membership_group": float(len(member_loss)),
        "benign_samples_per_group": float(counts[0]),
        "attack_samples_per_group": float(counts[1]),
    }

    alpha = 1.0 - confidence_level
    if bootstrap_repetitions > 0:
        rng = np.random.default_rng(seed + 97_409)
        bootstrap = {
            key: []
            for key in (
                "mia_loss_auc",
                "mia_advantage",
                "mia_tpr_at_fixed_fpr",
                "member_nonmember_loss_gap",
            )
        }
        member_selected_labels = member_labels[member_idx]
        nonmember_selected_labels = nonmember_labels[nonmember_idx]
        for _ in range(bootstrap_repetitions):
            member_parts = []
            nonmember_parts = []
            for label in (0, 1):
                member_candidates = np.flatnonzero(member_selected_labels == label)
                nonmember_candidates = np.flatnonzero(
                    nonmember_selected_labels == label
                )
                member_parts.append(
                    rng.choice(member_candidates, len(member_candidates), replace=True)
                )
                nonmember_parts.append(
                    rng.choice(
                        nonmember_candidates, len(nonmember_candidates), replace=True
                    )
                )
            measured = _loss_attack_statistics(
                member_loss[np.concatenate(member_parts)],
                nonmember_loss[np.concatenate(nonmember_parts)],
                fixed_fpr,
            )
            for key in bootstrap:
                bootstrap[key].append(measured[key])
        for key, values in bootstrap.items():
            lower, upper = np.quantile(values, [alpha / 2.0, 1.0 - alpha / 2.0])
            result[f"{key}_ci_lower"] = float(lower)
            result[f"{key}_ci_upper"] = float(upper)
    else:
        for key in (
            "mia_loss_auc",
            "mia_advantage",
            "mia_tpr_at_fixed_fpr",
            "member_nonmember_loss_gap",
        ):
            result[f"{key}_ci_lower"] = float("nan")
            result[f"{key}_ci_upper"] = float("nan")
    result["bootstrap_repetitions"] = float(bootstrap_repetitions)
    result["confidence_level"] = float(confidence_level)
    result.update(
        _learned_output_attack(
            member_labels[member_idx],
            member_probs,
            nonmember_labels[nonmember_idx],
            nonmember_probs,
            fixed_fpr,
            seed + 271,
        )
    )
    return result


def paired_loss_mia_auc_gap(
    member_labels: np.ndarray,
    candidate_member_probabilities: np.ndarray,
    reference_member_probabilities: np.ndarray,
    nonmember_labels: np.ndarray,
    candidate_nonmember_probabilities: np.ndarray,
    reference_nonmember_probabilities: np.ndarray,
    member_indices: np.ndarray,
    nonmember_indices: np.ndarray,
    fixed_fpr: float,
    bootstrap_repetitions: int,
    confidence_level: float,
    seed: int,
) -> dict[str, float]:
    """Paired bootstrap interval for candidate minus reference loss-MIA AUC."""

    member_indices = np.asarray(member_indices, dtype=np.int64)
    nonmember_indices = np.asarray(nonmember_indices, dtype=np.int64)
    selected_member_labels = member_labels[member_indices]
    selected_nonmember_labels = nonmember_labels[nonmember_indices]

    def losses(probabilities: np.ndarray, indices: np.ndarray, labels: np.ndarray):
        selected = np.clip(probabilities[indices], EPSILON, 1.0)
        return -np.log(selected[np.arange(len(indices)), labels])

    candidate_member_loss = losses(
        candidate_member_probabilities, member_indices, selected_member_labels
    )
    reference_member_loss = losses(
        reference_member_probabilities, member_indices, selected_member_labels
    )
    candidate_nonmember_loss = losses(
        candidate_nonmember_probabilities, nonmember_indices, selected_nonmember_labels
    )
    reference_nonmember_loss = losses(
        reference_nonmember_probabilities, nonmember_indices, selected_nonmember_labels
    )
    candidate_auc = _loss_attack_statistics(
        candidate_member_loss, candidate_nonmember_loss, fixed_fpr
    )["mia_loss_auc"]
    reference_auc = _loss_attack_statistics(
        reference_member_loss, reference_nonmember_loss, fixed_fpr
    )["mia_loss_auc"]
    result = {"mia_auc_gap_to_retrained": candidate_auc - reference_auc}
    if bootstrap_repetitions <= 0:
        result["mia_auc_gap_to_retrained_ci_lower"] = float("nan")
        result["mia_auc_gap_to_retrained_ci_upper"] = float("nan")
        return result

    rng = np.random.default_rng(seed)
    gaps = []
    for _ in range(bootstrap_repetitions):
        member_parts = []
        nonmember_parts = []
        for label in (0, 1):
            member_candidates = np.flatnonzero(selected_member_labels == label)
            nonmember_candidates = np.flatnonzero(selected_nonmember_labels == label)
            member_parts.append(
                rng.choice(member_candidates, len(member_candidates), replace=True)
            )
            nonmember_parts.append(
                rng.choice(
                    nonmember_candidates, len(nonmember_candidates), replace=True
                )
            )
        member_draw = np.concatenate(member_parts)
        nonmember_draw = np.concatenate(nonmember_parts)
        candidate_draw_auc = _loss_attack_statistics(
            candidate_member_loss[member_draw],
            candidate_nonmember_loss[nonmember_draw],
            fixed_fpr,
        )["mia_loss_auc"]
        reference_draw_auc = _loss_attack_statistics(
            reference_member_loss[member_draw],
            reference_nonmember_loss[nonmember_draw],
            fixed_fpr,
        )["mia_loss_auc"]
        gaps.append(candidate_draw_auc - reference_draw_auc)
    alpha = 1.0 - confidence_level
    lower, upper = np.quantile(gaps, [alpha / 2.0, 1.0 - alpha / 2.0])
    result["mia_auc_gap_to_retrained_ci_lower"] = float(lower)
    result["mia_auc_gap_to_retrained_ci_upper"] = float(upper)
    return result


## 8. Resource monitoring

**What the following block does:** This cell defines a context manager that synchronizes the selected device, measures wall time, samples process resident memory in a background thread, and records CUDA peak allocation when CUDA exposes that counter. The monitor reports both absolute peaks and increments relative to memory already held at entry; MPS peak allocation remains unavailable and is reported as zero. The training and unlearning functions additionally record optimizer steps, samples, rollback time, repair-optimization time, retraining-optimization time, and validation time. The primary speedup uses online rollback-plus-repair time versus scratch optimization time and excludes downstream metric evaluation. It also reports the total traced original-training time and trace storage, because online unlearning assumes that trace already exists in memory or on disk. It does not pretend that this total is the incremental tracing overhead; measuring that overhead would require a separate untraced training benchmark.

In [14]:
"""Wall-clock, CPU RSS, and accelerator peak-memory measurement."""

from __future__ import annotations

import os
import platform
import resource
import threading
import time
from dataclasses import dataclass

import torch



def _rss_bytes() -> int:
    try:
        import psutil

        return int(psutil.Process(os.getpid()).memory_info().rss)
    except ImportError:
        usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        # macOS reports bytes; Linux and most BSDs report KiB.
        return int(usage if platform.system() == "Darwin" else usage * 1024)


@dataclass
class ResourceUsage:
    elapsed_seconds: float
    rss_start_bytes: int
    rss_peak_bytes: int
    rss_peak_increment_bytes: int
    accelerator_start_bytes: int
    accelerator_peak_bytes: int
    accelerator_peak_increment_bytes: int


class ResourceMonitor:
    """Sample process RSS while an operation executes.

    The reported RSS increment is relative to memory already held at entry, so
    model/data allocations common to unlearning and retraining are not charged
    twice.  CUDA peak allocation is reset at entry.  MPS currently lacks an
    equivalent reliable peak counter and reports zero for accelerator peak.
    """

    def __init__(self, device: torch.device, sample_interval: float = 0.02) -> None:
        self.device = device
        self.sample_interval = sample_interval
        self.result: ResourceUsage | None = None
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self._start_rss = 0
        self._peak_rss = 0
        self._started = 0.0
        self._accelerator_start = 0

    def _sample(self) -> None:
        while not self._stop.wait(self.sample_interval):
            self._peak_rss = max(self._peak_rss, _rss_bytes())

    def __enter__(self) -> ResourceMonitor:
        synchronize(self.device)
        if self.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(self.device)
            self._accelerator_start = int(torch.cuda.memory_allocated(self.device))
        self._start_rss = _rss_bytes()
        self._peak_rss = self._start_rss
        self._stop.clear()
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()
        self._started = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> None:
        synchronize(self.device)
        elapsed = time.perf_counter() - self._started
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=max(1.0, self.sample_interval * 5))
        self._peak_rss = max(self._peak_rss, _rss_bytes())
        accelerator = 0
        if self.device.type == "cuda":
            accelerator = int(torch.cuda.max_memory_allocated(self.device))
        self.result = ResourceUsage(
            elapsed_seconds=elapsed,
            rss_start_bytes=self._start_rss,
            rss_peak_bytes=self._peak_rss,
            rss_peak_increment_bytes=max(0, self._peak_rss - self._start_rss),
            accelerator_start_bytes=self._accelerator_start,
            accelerator_peak_bytes=accelerator,
            accelerator_peak_increment_bytes=max(
                0, accelerator - self._accelerator_start
            ),
        )


## 9. Full leave-one-domain-out experiment

**What the following block does:** This is the orchestration cell. It saves the resolved configuration and software environment, prepares every domain once, and loops over architectures and seeds. For each pair it creates one initialization, trains the all-domain original while logging domain traces, and verifies that initialization plus the traces reconstructs the checkpoint. For every possible forgotten domain it independently rolls back and repairs the original, then constructs the scratch reference by explicitly loading the exact same initialization and training only on retained domains with the paired schedule. Because the strict schema and transform are data-independent and retained-domain sampling seeds do not depend on list order, that reference represents the configured pipeline with the selected domain absent.

All three models are evaluated on every test domain. The cell writes per-domain and retained-macro utility, similarity to scratch, matched membership attacks, individual and paired bootstrap intervals, optimization and resource costs, traced-training cost and trace storage, checkpoints, histories, and reproducibility manifests. Optional prediction archives include labels, source-record IDs, and the exact MIA indices. The final summary has one row per architecture, seed, and forgotten domain; its aggregate uses sample standard deviation, standard error, and Student-t 95% intervals across seeds.

In [15]:
"""End-to-end leave-one-NetFlow-domain-out unlearning experiment."""

from __future__ import annotations

import gc
import hashlib
import json
import math
import platform
import re
import sys
from collections.abc import Mapping, Sequence
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch


PERFORMANCE_KEYS = (
    "accuracy",
    "balanced_accuracy",
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "roc_auc",
    "average_precision",
    "log_loss",
)


def _safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_")


def _scenario_seed(base_seed: int, domain: str, purpose: str) -> int:
    payload = f"{base_seed}\x1f{domain}\x1f{purpose}".encode()
    return int.from_bytes(hashlib.blake2b(payload, digest_size=4).digest(), "little")


def _json_dump(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def _write_rows(path: Path, rows: Sequence[Mapping[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)


def _aggregate_summary_across_seeds(
    rows: Sequence[Mapping[str, object]],
) -> list[dict[str, object]]:
    frame = pd.DataFrame(rows)
    group_columns = ["architecture", "forgotten_domain"]
    value_columns = [
        column
        for column in frame.select_dtypes(include=[np.number]).columns
        if column != "seed"
    ]
    output: list[dict[str, object]] = []
    for keys, group in frame.groupby(group_columns, sort=False):
        row: dict[str, object] = {
            "architecture": keys[0],
            "forgotten_domain": keys[1],
            "seed_count": int(group["seed"].nunique()),
        }
        for column in value_columns:
            values = group[column].astype(float).to_numpy()
            row[f"{column}_mean"] = float(np.mean(values))
            count = len(values)
            if count >= 2:
                standard_deviation = float(np.std(values, ddof=1))
                standard_error = standard_deviation / math.sqrt(count)
            else:
                standard_deviation = float("nan")
                standard_error = float("nan")
            row[f"{column}_std"] = standard_deviation
            row[f"{column}_sem"] = standard_error
            # Student-t 95% critical values for the small seed counts normally
            # used here; normal approximation beyond 30 seeds.
            try:
                from scipy.stats import t

                critical = float(t.ppf(0.975, df=count - 1)) if count >= 2 else math.nan
            except ImportError:
                critical = 1.96 if count >= 2 else math.nan
            row[f"{column}_ci95_lower"] = float(
                np.mean(values) - critical * standard_error
            )
            row[f"{column}_ci95_upper"] = float(
                np.mean(values) + critical * standard_error
            )
        output.append(row)
    return output


def _save_checkpoint(
    path: Path,
    state_dict: Mapping[str, torch.Tensor],
    metadata: Mapping[str, object],
    domain_deltas: DomainDeltas | None = None,
    initial_state: Mapping[str, torch.Tensor] | None = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload: dict[str, object] = {
        "state_dict": dict(state_dict),
        "metadata": dict(metadata),
    }
    if domain_deltas is not None:
        payload["domain_deltas"] = domain_deltas
    if initial_state is not None:
        payload["initial_state_dict"] = dict(initial_state)
    torch.save(payload, path)


def _resource_dict(prefix: str, usage: ResourceUsage) -> dict[str, object]:
    return {
        f"{prefix}_wall_seconds": usage.elapsed_seconds,
        f"{prefix}_rss_start_bytes": usage.rss_start_bytes,
        f"{prefix}_rss_peak_bytes": usage.rss_peak_bytes,
        f"{prefix}_rss_peak_increment_bytes": usage.rss_peak_increment_bytes,
        f"{prefix}_accelerator_start_bytes": usage.accelerator_start_bytes,
        f"{prefix}_accelerator_peak_bytes": usage.accelerator_peak_bytes,
        f"{prefix}_accelerator_peak_increment_bytes": (
            usage.accelerator_peak_increment_bytes
        ),
    }


def _delta_reconstruction_error(
    initial: Mapping[str, torch.Tensor],
    final: Mapping[str, torch.Tensor],
    deltas: DomainDeltas,
) -> dict[str, float]:
    squared_error = 0.0
    squared_reference = 0.0
    max_abs = 0.0
    for name, start in initial.items():
        if name not in next(iter(deltas.values())):
            continue
        reconstructed = start.float().clone()
        for domain_delta in deltas.values():
            reconstructed.add_(domain_delta[name].float())
        target = final[name].float()
        error = reconstructed - target
        squared_error += float(torch.sum(error * error).item())
        squared_reference += float(torch.sum(target * target).item())
        if error.numel():
            max_abs = max(max_abs, float(error.abs().max().item()))
    l2 = math.sqrt(squared_error)
    return {
        "delta_reconstruction_l2": l2,
        "delta_reconstruction_relative_l2": l2
        / max(math.sqrt(squared_reference), 1e-12),
        "delta_reconstruction_max_abs": max_abs,
    }


def _aggregate_retained_rows(
    base: Mapping[str, object],
    model_name: str,
    per_domain_metrics: Mapping[str, Mapping[str, float]],
    retained: Sequence[str],
) -> dict[str, object]:
    row: dict[str, object] = dict(base)
    row.update(
        {
            "model": model_name,
            "evaluation_domain": "RETAINED_MACRO",
            "evaluation_scope": "retained_macro",
        }
    )
    for key in PERFORMANCE_KEYS:
        row[key] = float(np.mean([per_domain_metrics[name][key] for name in retained]))
    for key in (
        "true_negative",
        "false_positive",
        "false_negative",
        "true_positive",
        "samples",
    ):
        row[key] = float(np.sum([per_domain_metrics[name][key] for name in retained]))
    total_samples = sum(per_domain_metrics[name]["samples"] for name in retained)
    row["attack_rate"] = float(
        sum(
            per_domain_metrics[name]["attack_rate"]
            * per_domain_metrics[name]["samples"]
            for name in retained
        )
        / max(total_samples, 1.0)
    )
    return row


def _clear_accelerator(device: torch.device) -> None:
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()


def _training_metadata(result: TrainingResult) -> dict[str, object]:
    return {
        "elapsed_seconds": result.elapsed_seconds,
        "optimization_seconds": result.optimization_seconds,
        "validation_seconds": result.validation_seconds,
        "validation_evaluations": result.validation_evaluations,
        "optimizer_steps": result.optimizer_steps,
        "samples_seen": result.samples_seen,
        "best_epoch": result.best_epoch,
        "best_validation_loss": result.best_validation_loss,
        "checkpoint_bytes": result.checkpoint_bytes,
        "domain_delta_bytes": result.delta_bytes,
    }


def run_experiment(config: ExperimentConfig, run_dir: str | Path | None = None) -> Path:
    """Run both architectures and every requested forgotten domain."""

    config.validate()
    if run_dir is None:
        stamp = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
        destination = Path(config.runtime.output_dir) / stamp
    else:
        destination = Path(run_dir)
    destination = destination.expanduser().resolve()
    destination.mkdir(parents=True, exist_ok=False)
    config.to_json(destination / "resolved_config.json")

    device = resolve_device(config.runtime.device)
    environment = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "device": str(device),
        "cuda_device": (
            torch.cuda.get_device_name(device) if device.type == "cuda" else None
        ),
        "feature_schema_scope": (
            "predeclared" if config.data.common_features is not None else "observed"
        ),
        "transform_scope": (
            "stateless_data_independent"
            if config.data.scaler in {"fixed_log", "none"}
            else "learned_exploratory"
        ),
        "unlearning_scope": (
            "configured_model_pipeline"
            if config.data.common_features is not None
            and config.data.scaler in {"fixed_log", "none"}
            else "model_weights_only"
        ),
        "privacy_statement": (
            "Membership attacks are empirical audits, not proofs of privacy."
        ),
    }
    _json_dump(destination / "environment.json", environment)

    print(f"[data] preparing {len(config.data.datasets)} domains")
    prepared = prepare_data(config.data, destination / "data")
    domain_names = list(prepared.domains)
    print(
        f"[data] {prepared.input_dim} strict common features; domains: "
        + ", ".join(domain_names)
    )

    all_metric_rows: list[dict[str, object]] = []
    all_similarity_rows: list[dict[str, object]] = []
    all_mia_rows: list[dict[str, object]] = []
    all_efficiency_rows: list[dict[str, object]] = []
    all_summary_rows: list[dict[str, object]] = []

    for seed in config.training.seeds:
        for architecture in config.models.architectures:
            architecture_dir = destination / architecture / f"seed_{seed}"
            architecture_dir.mkdir(parents=True, exist_ok=True)
            print(f"\n[original] architecture={architecture} seed={seed}")

            # Seeding before construction makes scratch gold models use exactly
            # the same initialization as the original all-domain model.
            seed_everything(seed, config.runtime.deterministic)
            original_model = build_model(
                architecture, prepared.input_dim, config.models
            )
            initial_state = cpu_state_dict(original_model)
            with ResourceMonitor(device) as original_monitor:
                original_result = train_model(
                    original_model,
                    prepared.domains,
                    config.training,
                    device,
                    seed,
                    domain_names=domain_names,
                    track_domain_deltas=True,
                    deterministic=config.runtime.deterministic,
                )
            assert original_monitor.result is not None
            if original_result.domain_deltas is None:
                raise RuntimeError("Original training did not return domain deltas")
            delta_check = _delta_reconstruction_error(
                initial_state,
                original_result.state_dict,
                original_result.domain_deltas,
            )
            if delta_check["delta_reconstruction_relative_l2"] > 1e-4:
                raise RuntimeError(
                    "Per-domain update bookkeeping failed reconstruction check: "
                    f"{delta_check}"
                )
            original_metadata = {
                "architecture": architecture,
                "seed": seed,
                "input_dim": prepared.input_dim,
                "features": prepared.common_features,
                "training": _training_metadata(original_result),
                "delta_check": delta_check,
            }
            _save_checkpoint(
                architecture_dir / "original.pt",
                original_result.state_dict,
                original_metadata,
                domain_deltas=original_result.domain_deltas,
                initial_state=initial_state,
            )
            _write_rows(
                architecture_dir / "original_training_history.csv",
                original_result.history,
            )
            _json_dump(architecture_dir / "original_metadata.json", original_metadata)

            original_test_probs = {
                name: predict_proba(
                    original_model,
                    prepared.domains[name].x_test,
                    device,
                    config.training.batch_size * 4,
                )
                for name in domain_names
            }
            original_model.to("cpu")
            _clear_accelerator(device)
            original_train_probs: dict[str, np.ndarray] = {}

            for forget_domain in domain_names:
                retained = [name for name in domain_names if name != forget_domain]
                scenario_dir = architecture_dir / f"forget_{_safe_name(forget_domain)}"
                scenario_dir.mkdir(parents=True, exist_ok=True)
                print(
                    f"[forget] architecture={architecture} seed={seed} "
                    f"domain={forget_domain}"
                )

                # Every deletion starts from the same original checkpoint; no
                # deletion is applied sequentially to a previously unlearned model.
                seed_everything(seed, config.runtime.deterministic)
                unlearned_model = build_model(
                    architecture, prepared.input_dim, config.models
                )
                with ResourceMonitor(device) as unlearning_monitor:
                    unlearning_result = unlearn_domain(
                        unlearned_model,
                        original_result.state_dict,
                        original_result.domain_deltas,
                        forget_domain,
                        prepared.domains,
                        config.training,
                        config.unlearning,
                        device,
                        _scenario_seed(seed, forget_domain, "repair"),
                    )
                assert unlearning_monitor.result is not None
                unlearned_model.to("cpu")
                _clear_accelerator(device)

                # Gold standard: identical initialization and training protocol,
                # but the forgotten domain never enters optimization/validation.
                seed_everything(seed, config.runtime.deterministic)
                gold_model = build_model(
                    architecture, prepared.input_dim, config.models
                )
                # Do not merely rely on repeating the construction seed: load
                # the exact original initialization as the counterfactual start.
                load_state_dict(gold_model, initial_state)
                for name, value in gold_model.state_dict().items():
                    if not torch.equal(value.detach().cpu(), initial_state[name]):
                        raise RuntimeError(
                            f"Gold initialization differs at parameter {name}"
                        )
                with ResourceMonitor(device) as retraining_monitor:
                    retraining_result = train_model(
                        gold_model,
                        prepared.domains,
                        config.training,
                        device,
                        seed,
                        domain_names=retained,
                        track_domain_deltas=False,
                        deterministic=config.runtime.deterministic,
                    )
                assert retraining_monitor.result is not None
                gold_model.to("cpu")
                _clear_accelerator(device)

                _save_checkpoint(
                    scenario_dir / "unlearned.pt",
                    unlearning_result.state_dict,
                    {
                        "architecture": architecture,
                        "seed": seed,
                        "forgotten_domain": forget_domain,
                        "method": config.unlearning.method,
                        "elapsed_seconds": unlearning_result.elapsed_seconds,
                        "optimizer_steps": unlearning_result.optimizer_steps,
                        "samples_seen": unlearning_result.samples_seen,
                    },
                )
                _save_checkpoint(
                    scenario_dir / "retrained_gold.pt",
                    retraining_result.state_dict,
                    {
                        "architecture": architecture,
                        "seed": seed,
                        "forgotten_domain": forget_domain,
                        "retained_domains": retained,
                        "training": _training_metadata(retraining_result),
                    },
                )
                _write_rows(
                    scenario_dir / "retrained_training_history.csv",
                    retraining_result.history,
                )

                unlearned_test_probs = {
                    name: predict_proba(
                        unlearned_model,
                        prepared.domains[name].x_test,
                        device,
                        config.training.batch_size * 4,
                    )
                    for name in domain_names
                }
                unlearned_model.to("cpu")
                _clear_accelerator(device)
                retrained_test_probs = {
                    name: predict_proba(
                        gold_model,
                        prepared.domains[name].x_test,
                        device,
                        config.training.batch_size * 4,
                    )
                    for name in domain_names
                }
                gold_model.to("cpu")
                _clear_accelerator(device)
                test_probabilities: dict[str, dict[str, np.ndarray]] = {
                    "original": original_test_probs,
                    "unlearned": unlearned_test_probs,
                    "retrained": retrained_test_probs,
                }

                base: dict[str, object] = {
                    "architecture": architecture,
                    "seed": seed,
                    "forgotten_domain": forget_domain,
                }
                scenario_metric_rows: list[dict[str, object]] = []
                metric_lookup: dict[str, dict[str, dict[str, float]]] = {}
                for model_name, by_domain in test_probabilities.items():
                    metric_lookup[model_name] = {}
                    for evaluation_domain in domain_names:
                        measured = classification_metrics(
                            prepared.domains[evaluation_domain].y_test,
                            by_domain[evaluation_domain],
                        )
                        metric_lookup[model_name][evaluation_domain] = measured
                        row = dict(base)
                        row.update(
                            {
                                "model": model_name,
                                "evaluation_domain": evaluation_domain,
                                "evaluation_scope": (
                                    "forgotten"
                                    if evaluation_domain == forget_domain
                                    else "retained"
                                ),
                                **measured,
                            }
                        )
                        scenario_metric_rows.append(row)
                    scenario_metric_rows.append(
                        _aggregate_retained_rows(
                            base, model_name, metric_lookup[model_name], retained
                        )
                    )

                scenario_similarity_rows: list[dict[str, object]] = []
                for candidate in ("original", "unlearned"):
                    for evaluation_domain in domain_names:
                        similarity = prediction_similarity(
                            test_probabilities[candidate][evaluation_domain],
                            test_probabilities["retrained"][evaluation_domain],
                        )
                        candidate_metrics = metric_lookup[candidate][evaluation_domain]
                        gold_metrics = metric_lookup["retrained"][evaluation_domain]
                        row = dict(base)
                        row.update(
                            {
                                "candidate_model": candidate,
                                "evaluation_domain": evaluation_domain,
                                "evaluation_scope": (
                                    "forgotten"
                                    if evaluation_domain == forget_domain
                                    else "retained"
                                ),
                                **similarity,
                                "f1_gap_to_retrained": candidate_metrics["f1_attack"]
                                - gold_metrics["f1_attack"],
                                "roc_auc_gap_to_retrained": candidate_metrics["roc_auc"]
                                - gold_metrics["roc_auc"],
                                "log_loss_gap_to_retrained": candidate_metrics[
                                    "log_loss"
                                ]
                                - gold_metrics["log_loss"],
                            }
                        )
                        scenario_similarity_rows.append(row)

                if forget_domain not in original_train_probs:
                    original_train_probs[forget_domain] = predict_proba(
                        original_model,
                        prepared.domains[forget_domain].x_train,
                        device,
                        config.training.batch_size * 4,
                    )
                    original_model.to("cpu")
                    _clear_accelerator(device)
                unlearned_train_probabilities = predict_proba(
                    unlearned_model,
                    prepared.domains[forget_domain].x_train,
                    device,
                    config.training.batch_size * 4,
                )
                unlearned_model.to("cpu")
                _clear_accelerator(device)
                retrained_train_probabilities = predict_proba(
                    gold_model,
                    prepared.domains[forget_domain].x_train,
                    device,
                    config.training.batch_size * 4,
                )
                gold_model.to("cpu")
                _clear_accelerator(device)
                train_probabilities = {
                    "original": original_train_probs[forget_domain],
                    "unlearned": unlearned_train_probabilities,
                    "retrained": retrained_train_probabilities,
                }
                scenario_mia_rows: list[dict[str, object]] = []
                member_indices, nonmember_indices, _ = matched_membership_indices(
                    prepared.domains[forget_domain].y_train,
                    prepared.domains[forget_domain].y_test,
                    config.attack.max_samples_per_class,
                    _scenario_seed(seed, forget_domain, "mia-sample"),
                )
                for model_name in ("original", "unlearned", "retrained"):
                    attack = loss_membership_attack(
                        prepared.domains[forget_domain].y_train,
                        train_probabilities[model_name],
                        prepared.domains[forget_domain].y_test,
                        test_probabilities[model_name][forget_domain],
                        config.attack.max_samples_per_class,
                        config.attack.fixed_fpr,
                        _scenario_seed(seed, forget_domain, "mia-bootstrap"),
                        bootstrap_repetitions=config.attack.bootstrap_repetitions,
                        confidence_level=config.attack.confidence_level,
                        member_indices=member_indices,
                        nonmember_indices=nonmember_indices,
                    )
                    row = dict(base)
                    row.update({"model": model_name, **attack})
                    scenario_mia_rows.append(row)
                mia_rows_by_model = {
                    str(row["model"]): row for row in scenario_mia_rows
                }
                mia_rows_by_model["retrained"].update(
                    {
                        "mia_auc_gap_to_retrained": 0.0,
                        "mia_auc_gap_to_retrained_ci_lower": 0.0,
                        "mia_auc_gap_to_retrained_ci_upper": 0.0,
                    }
                )
                for model_name in ("original", "unlearned"):
                    paired_gap = paired_loss_mia_auc_gap(
                        prepared.domains[forget_domain].y_train,
                        train_probabilities[model_name],
                        train_probabilities["retrained"],
                        prepared.domains[forget_domain].y_test,
                        test_probabilities[model_name][forget_domain],
                        test_probabilities["retrained"][forget_domain],
                        member_indices,
                        nonmember_indices,
                        config.attack.fixed_fpr,
                        config.attack.bootstrap_repetitions,
                        config.attack.confidence_level,
                        _scenario_seed(seed, forget_domain, "mia-paired-gap"),
                    )
                    mia_rows_by_model[model_name].update(paired_gap)

                efficiency: dict[str, object] = {
                    **base,
                    "device": str(device),
                    "parameter_count": parameter_count(original_model),
                    "checkpoint_bytes": original_result.checkpoint_bytes,
                    "domain_delta_store_bytes": original_result.delta_bytes,
                    "per_domain_delta_bytes": state_nbytes(
                        original_result.domain_deltas[forget_domain]
                    ),
                    "original_training_optimizer_steps": (
                        original_result.optimizer_steps
                    ),
                    "original_training_samples_seen": original_result.samples_seen,
                    "unlearning_optimizer_steps": unlearning_result.optimizer_steps,
                    "unlearning_samples_seen": unlearning_result.samples_seen,
                    "full_retraining_optimizer_steps": (
                        retraining_result.optimizer_steps
                    ),
                    "full_retraining_samples_seen": retraining_result.samples_seen,
                    "rollback_l2": unlearning_result.rollback_l2,
                    "rollback_max_abs": unlearning_result.rollback_max_abs,
                    "unlearning_rollback_seconds": (unlearning_result.rollback_seconds),
                    "unlearning_repair_optimization_seconds": (
                        unlearning_result.repair_optimization_seconds
                    ),
                    "unlearning_online_seconds": unlearning_result.elapsed_seconds,
                    "full_retraining_optimization_seconds": (
                        retraining_result.optimization_seconds
                    ),
                    "full_retraining_validation_seconds": (
                        retraining_result.validation_seconds
                    ),
                    "original_traced_training_optimization_seconds": (
                        original_result.optimization_seconds
                    ),
                    "trace_storage_bytes": original_result.delta_bytes,
                    "incremental_trace_creation_overhead_measured": False,
                    "timing_scope_note": (
                        "Online unlearning assumes the original checkpoint and "
                        "domain-update trace are already resident. The total traced "
                        "original-training time and trace storage are reported, but "
                        "incremental tracing overhead versus an untraced training "
                        "run is not estimated. Speedup uses optimization time on "
                        "both sides and excludes downstream metric evaluation."
                    ),
                    **delta_check,
                    **_resource_dict("original_training", original_monitor.result),
                    **_resource_dict("unlearning", unlearning_monitor.result),
                    **_resource_dict("full_retraining", retraining_monitor.result),
                }
                efficiency["online_speedup_vs_retraining_optimization"] = float(
                    retraining_result.optimization_seconds
                    / max(unlearning_result.elapsed_seconds, 1e-12)
                )
                efficiency["optimizer_step_reduction_fraction"] = float(
                    1.0
                    - unlearning_result.optimizer_steps
                    / max(retraining_result.optimizer_steps, 1)
                )
                efficiency["sample_reduction_fraction"] = float(
                    1.0
                    - unlearning_result.samples_seen
                    / max(retraining_result.samples_seen, 1)
                )

                retained_macro = {
                    row["model"]: row
                    for row in scenario_metric_rows
                    if row["evaluation_scope"] == "retained_macro"
                }
                forgotten_rows = {
                    row["model"]: row
                    for row in scenario_metric_rows
                    if row["evaluation_scope"] == "forgotten"
                }
                mia_lookup = {row["model"]: row for row in scenario_mia_rows}
                forgotten_similarity = next(
                    row
                    for row in scenario_similarity_rows
                    if row["candidate_model"] == "unlearned"
                    and row["evaluation_domain"] == forget_domain
                )
                summary: dict[str, object] = dict(base)
                for model_name in ("original", "unlearned", "retrained"):
                    summary[f"{model_name}_retained_macro_f1"] = retained_macro[
                        model_name
                    ]["f1_attack"]
                    summary[f"{model_name}_retained_macro_roc_auc"] = retained_macro[
                        model_name
                    ]["roc_auc"]
                    summary[f"{model_name}_forgotten_f1"] = forgotten_rows[model_name][
                        "f1_attack"
                    ]
                    summary[f"{model_name}_forgotten_roc_auc"] = forgotten_rows[
                        model_name
                    ]["roc_auc"]
                    summary[f"{model_name}_mia_auc"] = mia_lookup[model_name][
                        "mia_loss_auc"
                    ]
                summary.update(
                    {
                        "unlearned_forgotten_prediction_agreement_with_retrained": (
                            forgotten_similarity["prediction_agreement"]
                        ),
                        "unlearned_forgotten_js_divergence_from_retrained": (
                            forgotten_similarity["mean_jensen_shannon_divergence"]
                        ),
                        "unlearned_forgotten_abs_f1_gap_from_retrained": abs(
                            float(forgotten_similarity["f1_gap_to_retrained"])
                        ),
                        "unlearned_abs_mia_auc_gap_from_retrained": abs(
                            float(mia_lookup["unlearned"]["mia_auc_gap_to_retrained"])
                        ),
                        "unlearning_online_seconds": unlearning_result.elapsed_seconds,
                        "full_retraining_optimization_seconds": (
                            retraining_result.optimization_seconds
                        ),
                        "online_speedup_vs_retraining_optimization": efficiency[
                            "online_speedup_vs_retraining_optimization"
                        ],
                    }
                )

                _write_rows(
                    scenario_dir / "classification_metrics.csv", scenario_metric_rows
                )
                _write_rows(
                    scenario_dir / "gold_similarity.csv", scenario_similarity_rows
                )
                _write_rows(
                    scenario_dir / "membership_inference.csv", scenario_mia_rows
                )
                _json_dump(scenario_dir / "efficiency.json", efficiency)
                _json_dump(scenario_dir / "summary.json", summary)
                if config.runtime.save_predictions:
                    arrays: dict[str, np.ndarray] = {}
                    for model_name, by_domain in test_probabilities.items():
                        for domain_name, probabilities in by_domain.items():
                            arrays[f"{model_name}__{_safe_name(domain_name)}__test"] = (
                                probabilities
                            )
                    for model_name, probabilities in train_probabilities.items():
                        arrays[f"{model_name}__{_safe_name(forget_domain)}__train"] = (
                            probabilities
                        )
                    forgotten_split = prepared.domains[forget_domain]
                    arrays["forgotten_train_record_ids"] = (
                        forgotten_split.train_record_ids
                    )
                    arrays["forgotten_test_record_ids"] = (
                        forgotten_split.test_record_ids
                    )
                    arrays["forgotten_train_labels"] = forgotten_split.y_train
                    arrays["forgotten_test_labels"] = forgotten_split.y_test
                    arrays["mia_member_indices"] = member_indices
                    arrays["mia_nonmember_indices"] = nonmember_indices
                    np.savez_compressed(scenario_dir / "predictions.npz", **arrays)

                all_metric_rows.extend(scenario_metric_rows)
                all_similarity_rows.extend(scenario_similarity_rows)
                all_mia_rows.extend(scenario_mia_rows)
                all_efficiency_rows.append(efficiency)
                all_summary_rows.append(summary)

                del unlearned_model, gold_model, test_probabilities, train_probabilities
                _clear_accelerator(device)

            del original_model, original_test_probs, original_train_probs
            _clear_accelerator(device)

    _write_rows(destination / "all_classification_metrics.csv", all_metric_rows)
    _write_rows(destination / "all_gold_similarity.csv", all_similarity_rows)
    _write_rows(destination / "all_membership_inference.csv", all_mia_rows)
    _write_rows(destination / "all_efficiency.csv", all_efficiency_rows)
    _write_rows(destination / "summary.csv", all_summary_rows)
    _write_rows(
        destination / "summary_aggregated.csv",
        _aggregate_summary_across_seeds(all_summary_rows),
    )
    _json_dump(
        destination / "completed.json",
        {
            "completed_utc": datetime.now(timezone.utc).isoformat(),
            "architectures": config.models.architectures,
            "seeds": config.training.seeds,
            "forgotten_domains": domain_names,
            "scenarios": len(all_summary_rows),
        },
    )
    print(f"\n[done] results written to {destination}")
    return destination


## 10. Tiny synthetic domains for an end-to-end smoke test

**What the following block does:** This cell creates four small artificial NetFlow-like CSV files so the entire workflow can be tested without downloading the research corpora. The domains share ten declared behavioural fields, contain controlled distribution shifts, and express labels as integers, text, Booleans, and attack-family names. Address and port columns exist so grouped splitting and identifier exclusion are exercised; an extra domain-only column confirms that undeclared fields never enter the model. The smoke configuration still uses grouped splits, stateless preprocessing, fixed loss weights, paired plain-SGD schedules, both architectures, every forget case, both privacy attacks, and artifact generation, but uses one seed, smaller models, fewer epochs, and fewer bootstrap repetitions. It validates plumbing only—its metrics are not scientific evidence.

In [16]:
"""Small four-domain generator used only to smoke-test the full pipeline."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd



def make_synthetic_config(root: str | Path) -> ExperimentConfig:
    root = Path(root).resolve()
    data_dir = root / "synthetic_domains"
    data_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(42)
    specs = []
    names = ["NF-SYNTH-A", "NF-SYNTH-B", "NF-SYNTH-C", "NF-SYNTH-D"]
    for domain_index, name in enumerate(names):
        rows = 320
        protocol = rng.integers(0, 4, size=rows)
        packets = rng.poisson(12 + domain_index, size=rows) + 1
        in_bytes = packets * rng.lognormal(4.0 + 0.08 * domain_index, 0.5, rows)
        out_bytes = packets * rng.lognormal(3.6 - 0.04 * domain_index, 0.6, rows)
        duration = rng.lognormal(4.0 + 0.1 * domain_index, 0.7, rows)
        latent = (
            0.025 * packets
            + 0.0006 * in_bytes
            - 0.0004 * out_bytes
            + 0.18 * (protocol == 3)
            + 0.12 * domain_index
            + rng.normal(0, 0.5, rows)
        )
        threshold = np.quantile(latent, 0.65)
        labels = (latent > threshold).astype(np.int64)
        frame = pd.DataFrame(
            {
                "IPV4_SRC_ADDR": [
                    f"10.{domain_index}.0.{i % 255}" for i in range(rows)
                ],
                "IPV4_DST_ADDR": [
                    f"172.16.{domain_index}.{i % 255}" for i in range(rows)
                ],
                "L4_SRC_PORT": rng.integers(1_024, 65_535, rows),
                "L4_DST_PORT": rng.integers(1, 65_535, rows),
                "PROTOCOL": protocol,
                "IN_BYTES": in_bytes,
                "IN_PKTS": packets,
                "OUT_BYTES": out_bytes,
                "OUT_PKTS": rng.poisson(9, size=rows) + 1,
                "TCP_FLAGS": rng.integers(0, 32, rows),
                "FLOW_DURATION_MILLISECONDS": duration,
                "SRC_TO_DST_IAT_MAX": rng.lognormal(2.0, 0.8, rows),
                "DST_TO_SRC_IAT_MAX": rng.lognormal(1.8, 0.9, rows),
                f"DOMAIN_ONLY_{domain_index}": rng.normal(size=rows),
            }
        )
        if domain_index == 0:
            frame["Label"] = labels
        elif domain_index == 1:
            frame["LABEL"] = np.where(labels == 1, "Attack", "Benign")
        elif domain_index == 2:
            frame["label"] = labels.astype(bool)
        else:
            frame["Label"] = np.where(labels == 1, "DDoS", "Normal")
        path = data_dir / f"{name}.csv"
        frame.to_csv(path, index=False)
        specs.append(DatasetConfig(name=name, path=str(path), sample_rows=None))

    return ExperimentConfig(
        data=DataConfig(
            datasets=specs,
            common_features=[
                "PROTOCOL",
                "IN_BYTES",
                "IN_PKTS",
                "OUT_BYTES",
                "OUT_PKTS",
                "TCP_FLAGS",
                "FLOW_DURATION",
                "SRC_TO_DST_IAT_MAX",
                "DST_TO_SRC_IAT_MAX",
                "BYTES_PER_PKT",
            ],
            sample_rows_per_dataset=None,
            csv_chunk_rows=1_000,
            scaler="fixed_log",
            scaler_fit_rows=None,
            allow_kaggle_download=False,
            split_strategy="group_stratified",
            hash_source_files=False,
            strict_protocol=False,
        ),
        models=ModelConfig(
            architectures=["mlp", "tabtransformer"],
            latent_dim=8,
            mlp_hidden_dims=[16],
            dropout=0.0,
            tab_d_token=8,
            tab_heads=2,
            tab_layers=1,
            tab_ffn_factor=2,
        ),
        training=TrainingConfig(
            epochs=2,
            batch_size=64,
            learning_rate=2e-3,
            patience=2,
            seeds=[42],
            strict_unlearning_protocol=False,
        ),
        unlearning=UnlearningConfig(
            repair_epochs=1,
            repair_fraction=0.25,
            repair_learning_rate=5e-4,
        ),
        attack=AttackConfig(
            max_samples_per_class=30,
            fixed_fpr=0.05,
            bootstrap_repetitions=25,
        ),
        runtime=RuntimeConfig(
            output_dir=str(root / "synthetic_results"),
            device="cpu",
            deterministic=True,
        ),
    )


## 11. Edit the run here

**What the following block does:** This is the only cell most students need to edit. `MODE = "smoke"` generates tiny local data; `"quick"` uses the real files with a 10,000-row cap, two epochs, one seed, and fewer bootstrap repetitions for debugging; `"full"` runs the publication protocol on the configured sample cap with at least three seeds. The four `DatasetConfig` entries identify local filename patterns and optional Kaggle fallbacks. If a corpus is intentionally split across files, set `allow_multiple_files=True` only for that entry; if a CSV is not UTF-8, state its true encoding explicitly rather than relying on a fallback.

The real configuration uses the public 48-feature list, grouped 70/15/15 partitions, duplicate removal, source hashes, stateless fixed-log transformation, both architectures, plain SGD, fixed loss weights, rollback and retained repair, matched attacks, and reproducible artifact output. Quick mode turns off only the strict publication validator because one seed cannot support across-seed inference; it retains the same core preprocessing and optimizer choices. The printed values let the student verify the run before starting expensive work.

In [17]:
from pathlib import Path
import tempfile

# --------------------------- EDIT THESE VALUES ---------------------------
MODE = "quick"  # "smoke", "quick", or "full"
DATA_DIR = Path("datasets").resolve()
ALLOW_KAGGLE_DOWNLOAD = True
SAMPLE_ROWS_PER_DATASET = 250_000  # fixed uniform cap; do not call this all rows
SEEDS = [42, 1337, 2026]  # full mode requires at least three independent seeds
DEVICE = "auto"  # "auto", "cpu", "cuda", or "mps"
SAVE_PREDICTIONS = False
CSV_ENCODING = "utf-8"
# ------------------------------------------------------------------------


def make_real_config() -> ExperimentConfig:
    datasets = [
        DatasetConfig(
            name="NF-UNSW-NB15",
            path=str(DATA_DIR),
            file_pattern="**/*UNSW*N*15*v3*",
            kaggle_slug="seyhed/nf-unsw-nb15-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
        DatasetConfig(
            name="NF-ToN-IoT",
            path=str(DATA_DIR),
            file_pattern="**/*ToN*IoT*v3*",
            kaggle_slug="seyhed/nf-ton-iot-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
        DatasetConfig(
            name="NF-BoT-IoT",
            path=str(DATA_DIR),
            file_pattern="**/*BoT*IoT*v3*",
            kaggle_slug="seyhed/nf-bot-iot-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
        DatasetConfig(
            name="NF-CSE-CIC-IDS2018",
            path=str(DATA_DIR),
            file_pattern="**/*CIC*IDS2018*v3*",
            kaggle_slug="seyhed/nf-cicids2018-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
    ]
    return ExperimentConfig(
        data=DataConfig(
            datasets=datasets,
            common_features=PUBLIC_V3_MODEL_FEATURES.copy(),
            train_fraction=0.70,
            validation_fraction=0.15,
            test_fraction=0.15,
            seed=42,
            sample_rows_per_dataset=SAMPLE_ROWS_PER_DATASET,
            csv_chunk_rows=100_000,
            scaler="fixed_log",
            scaler_fit_rows=None,
            allow_kaggle_download=ALLOW_KAGGLE_DOWNLOAD,
            split_strategy="group_stratified",
            group_columns=["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "PROTOCOL"],
            drop_exact_duplicates=True,
            hash_source_files=True,
            strict_protocol=True,
        ),
        models=ModelConfig(
            architectures=["mlp", "tabtransformer"],
            latent_dim=32,
            mlp_hidden_dims=[128, 64],
            dropout=0.10,
            tab_d_token=16,
            tab_heads=4,
            tab_layers=2,
            tab_ffn_factor=4,
        ),
        training=TrainingConfig(
            epochs=20,
            batch_size=256,
            learning_rate=1e-2,
            optimizer="sgd",
            sgd_momentum=0.0,
            weight_decay=0.0,
            patience=5,
            min_delta=1e-4,
            fixed_class_weights=[1.0, 1.0],
            gradient_clip_norm=5.0,
            domain_sampling="proportional",
            checkpoint_selection="final",
            seeds=SEEDS,
            strict_unlearning_protocol=True,
        ),
        unlearning=UnlearningConfig(
            method="amnesiac_rollback_repair",
            rollback_scale=1.0,
            repair_epochs=2,
            repair_fraction=0.25,
            repair_learning_rate=2e-3,
        ),
        attack=AttackConfig(
            max_samples_per_class=10_000,
            fixed_fpr=0.01,
            bootstrap_repetitions=1_000,
            confidence_level=0.95,
        ),
        runtime=RuntimeConfig(
            output_dir=str(Path("artifacts/netflow_unlearning").resolve()),
            device=DEVICE,
            save_predictions=SAVE_PREDICTIONS,
            deterministic=True,
        ),
    )


if MODE == "smoke":
    smoke_root = Path(tempfile.mkdtemp(prefix="netflow_unlearning_notebook_"))
    config = make_synthetic_config(smoke_root)
elif MODE in {"quick", "full"}:
    config = make_real_config()
    if MODE == "quick":
        config.data.sample_rows_per_dataset = 10_000
        config.data.hash_source_files = False
        config.data.strict_protocol = False
        config.training.epochs = 2
        config.training.seeds = config.training.seeds[:1]
        config.training.strict_unlearning_protocol = False
        config.unlearning.repair_epochs = 1
        config.attack.max_samples_per_class = 1_000
        config.attack.bootstrap_repetitions = 50
else:
    raise ValueError("MODE must be 'smoke', 'quick', or 'full'")

config.validate()
print(f"Mode: {MODE}")
print(f"Architectures: {config.models.architectures}")
print(f"Datasets: {[dataset.name for dataset in config.data.datasets]}")
print(f"Seeds: {config.training.seeds}")
print(f"Device request: {config.runtime.device}")
print(f"Strict publication validator: {config.data.strict_protocol}")


Mode: quick
Architectures: ['mlp', 'tabtransformer']
Datasets: ['NF-UNSW-NB15', 'NF-ToN-IoT', 'NF-BoT-IoT', 'NF-CSE-CIC-IDS2018']
Seeds: [42]
Device request: auto
Strict publication validator: False


## 12. Run every forget experiment

**What the following block does:** These two lines launch the computation. `run_experiment(config)` prepares data, trains one original model per architecture and seed, runs one independent rollback-and-repair deletion per domain, performs the paired scratch retraining for each deletion, evaluates utility and privacy, measures efficiency, and writes artifacts after every scenario. A full run contains many neural-network trainings and may take hours depending on dataset size and hardware. The returned absolute directory is assigned to `RESULT_DIR` and printed, so the exact checkpoints, manifests, detailed CSV files, and summaries can be found later.

In [18]:
RESULT_DIR = run_experiment(config)
print(f"Completed run: {RESULT_DIR}")

[data] preparing 4 domains


100%|██████████| 112M/112M [00:01<00:00, 60.7MB/s]

Extracting files...


100%|██████████| 433M/433M [00:05<00:00, 89.1MB/s]

Extracting files...


100%|██████████| 75.2M/75.2M [00:01<00:00, 59.5MB/s]

Extracting files...


100%|██████████| 801M/801M [00:17<00:00, 48.8MB/s]

Extracting files...


[data] 48 strict common features; domains: NF-UNSW-NB15, NF-ToN-IoT, NF-BoT-IoT, NF-CSE-CIC-IDS2018

[original] architecture=mlp seed=42
[forget] architecture=mlp seed=42 domain=NF-UNSW-NB15
[forget] architecture=mlp seed=42 domain=NF-ToN-IoT
[forget] architecture=mlp seed=42 domain=NF-BoT-IoT
[forget] architecture=mlp seed=42 domain=NF-CSE-CIC-IDS2018

[original] architecture=tabtransformer seed=42
[forget] architecture=tabtransformer seed=42 domain=NF-UNSW-NB15
[forget] architecture=tabtransformer seed=42 domain=NF-ToN-IoT
[forget] architecture=tabtransformer seed=42 domain=NF-BoT-IoT
[forget] architecture=tabtransformer seed=42 domain=NF-CSE-CIC-IDS2018

[done] results written to /content/artifacts/netflow_unlearning/run_20260905T130834Z
Completed run: /content/artifacts/netflow_unlearning/run_20260905T130834Z


## 13. Inspect the research tables

**What the following block does:** This cell loads the two highest-level tables. `summary.csv` contains one row for every architecture, seed, and forgotten-domain scenario. `summary_aggregated.csv` groups those rows and reports the mean, sample standard deviation (`ddof=1`), standard error, and Student-t 95% interval for every numeric outcome. With the one-seed smoke or quick run, spread and interval columns are intentionally `NaN` because uncertainty cannot be estimated from one observation. Draw conclusions from full multi-seed runs and inspect the detailed classification, similarity, membership, and efficiency files whenever a summary value is surprising.

In [19]:
summary = pd.read_csv(RESULT_DIR / "summary.csv")
summary_aggregated = pd.read_csv(RESULT_DIR / "summary_aggregated.csv")

display(summary)
display(summary_aggregated)

,architecture,seed,forgotten_domain,original_retained_macro_f1,original_retained_macro_roc_auc,original_forgotten_f1,original_forgotten_roc_auc,original_mia_auc,unlearned_retained_macro_f1,unlearned_retained_macro_roc_auc,...,retrained_forgotten_f1,retrained_forgotten_roc_auc,retrained_mia_auc,unlearned_forgotten_prediction_agreement_with_retrained,unlearned_forgotten_js_divergence_from_retrained,unlearned_forgotten_abs_f1_gap_from_retrained,unlearned_abs_mia_auc_gap_from_retrained,unlearning_online_seconds,full_retraining_optimization_seconds,online_speedup_vs_retraining_optimization
0,mlp,42,NF-UNSW-NB15,0.272853,0.872136,0.000000,0.785956,0.430005,0.615844,0.613769,...,0.000000,0.236437,0.418925,0.000000,0.279569,0.098672,0.021269,0.230965,1.441528,6.241339
1,mlp,42,NF-ToN-IoT,0.272853,0.846833,0.000000,0.861867,0.479000,0.250233,0.772185,...,0.078873,0.480039,0.462989,0.912958,0.093749,0.078873,0.007324,0.156745,1.102109,7.031207
2,mlp,42,NF-BoT-IoT,0.000000,0.871371,0.818559,0.788250,0.730985,0.000000,0.705881,...,0.000000,0.116347,0.554707,1.000000,0.022487,0.000000,0.076451,0.205140,1.609909,7.847845
3,mlp,42,NF-CSE-CIC-IDS2018,0.272853,0.812024,0.000000,0.966291,0.402347,0.548432,0.823157,...,0.005780,0.793749,0.347209,0.099602,0.302196,0.224537,0.195749,0.150116,1.084327,7.223275
4,tabtransformer,42,NF-UNSW-NB15,0.000000,0.674344,0.000000,0.910949,0.469981,0.592314,0.222260,...,0.000000,0.769024,0.573168,0.000000,0.274593,0.098672,0.172691,4.555758,39.413012,8.651252
5,tabtransformer,42,NF-ToN-IoT,0.000000,0.839664,0.000000,0.414989,0.481277,0.248397,0.502797,...,0.000000,0.219883,0.496142,0.984293,0.112344,0.016639,0.049913,4.507930,39.350981,8.729279
6,tabtransformer,42,NF-BoT-IoT,0.000000,0.659177,0.000000,0.956450,0.729692,0.000000,0.336002,...,0.000000,0.956960,0.466267,1.000000,0.011322,0.000000,0.043181,4.513991,39.610306,8.775007
7,tabtransformer,42,NF-CSE-CIC-IDS2018,0.000000,0.760796,0.000000,0.651593,0.335422,0.548432,0.606465,...,0.005650,0.705122,0.353808,0.104914,0.473661,0.224668,0.339144,5.876995,39.260355,6.680345


,architecture,forgotten_domain,seed_count,original_retained_macro_f1_mean,original_retained_macro_f1_std,original_retained_macro_f1_sem,original_retained_macro_f1_ci95_lower,original_retained_macro_f1_ci95_upper,original_retained_macro_roc_auc_mean,original_retained_macro_roc_auc_std,...,full_retraining_optimization_seconds_mean,full_retraining_optimization_seconds_std,full_retraining_optimization_seconds_sem,full_retraining_optimization_seconds_ci95_lower,full_retraining_optimization_seconds_ci95_upper,online_speedup_vs_retraining_optimization_mean,online_speedup_vs_retraining_optimization_std,online_speedup_vs_retraining_optimization_sem,online_speedup_vs_retraining_optimization_ci95_lower,online_speedup_vs_retraining_optimization_ci95_upper
0,mlp,NF-UNSW-NB15,1,0.272853,NaN,NaN,NaN,NaN,0.872136,NaN,...,1.441528,NaN,NaN,NaN,NaN,6.241339,NaN,NaN,NaN,NaN
1,mlp,NF-ToN-IoT,1,0.272853,NaN,NaN,NaN,NaN,0.846833,NaN,...,1.102109,NaN,NaN,NaN,NaN,7.031207,NaN,NaN,NaN,NaN
2,mlp,NF-BoT-IoT,1,0.000000,NaN,NaN,NaN,NaN,0.871371,NaN,...,1.609909,NaN,NaN,NaN,NaN,7.847845,NaN,NaN,NaN,NaN
3,mlp,NF-CSE-CIC-IDS2018,1,0.272853,NaN,NaN,NaN,NaN,0.812024,NaN,...,1.084327,NaN,NaN,NaN,NaN,7.223275,NaN,NaN,NaN,NaN
4,tabtransformer,NF-UNSW-NB15,1,0.000000,NaN,NaN,NaN,NaN,0.674344,NaN,...,39.413012,NaN,NaN,NaN,NaN,8.651252,NaN,NaN,NaN,NaN
5,tabtransformer,NF-ToN-IoT,1,0.000000,NaN,NaN,NaN,NaN,0.839664,NaN,...,39.350981,NaN,NaN,NaN,NaN,8.729279,NaN,NaN,NaN,NaN
6,tabtransformer,NF-BoT-IoT,1,0.000000,NaN,NaN,NaN,NaN,0.659177,NaN,...,39.610306,NaN,NaN,NaN,NaN,8.775007,NaN,NaN,NaN,NaN
7,tabtransformer,NF-CSE-CIC-IDS2018,1,0.000000,NaN,NaN,NaN,NaN,0.760796,NaN,...,39.260355,NaN,NaN,NaN,NaN,6.680345,NaN,NaN,NaN,NaN


## Interpretation reminder

A forgotten dataset's accuracy need not fall to chance: a model trained on other NetFlow domains can legitimately generalize to its attacks. The relevant target is whether the unlearned model approaches scratch retraining on the forgotten domain while preserving retained-domain utility. Judge this jointly from probability similarity, utility gaps, both membership attacks and their uncertainty, multiple random seeds, and online cost versus scratch optimization. The update rollback is an approximation because training steps interact, and the privacy tests are attacks rather than proofs. Therefore use language such as “no residual membership signal was detected by these attacks” instead of “privacy is guaranteed.” In strict mode, the declared schema and stateless transform remove the earlier preprocessing-contamination problem, so the comparison covers the configured learned model pipeline rather than only a scaler-contaminated weight comparison.

### Primary references

- Sarhan et al., [NetFlow datasets and standardized feature work](https://staff.itee.uq.edu.au/marius/NIDS_datasets/)
- Graves et al., [Amnesiac Machine Learning](https://arxiv.org/abs/2010.10981)
- Gorishniy et al., [Revisiting Deep Learning Models for Tabular Data (FT-Transformer)](https://arxiv.org/abs/2106.11959)
- Huang et al., [TabTransformer](https://arxiv.org/abs/2012.06678)
- Shokri et al., [Membership Inference Attacks Against Machine Learning Models](https://arxiv.org/abs/1610.05820)
